# 🤖 Chatbot Académico EPIIS-UNSAAC
## IF651 — Inteligencia Artificial | Proyecto Semestral 2026-1 | Segunda Entrega

| Campo | Detalle |
|---|---|
| **Asignatura** | IF651 Inteligencia Artificial |
| **Docentes** | Maritza Katherine Irpanocca Cusimayta / Jisbaj Gamarra Salas |
| **Integrantes** | Castro Pari Rayneld Fidel · Merma Ccarhuarupay Adel Alejandro · Florez Vega Yamir Wagner · Mamani Flores Natan |
| **Atributo** | AG08 Análisis de problemas |
| **Paradigma** | POO — Recuperación basada en corpus + scoring ponderado de keywords |

---
### ▶ Instrucciones de ejecución
1. **Entorno de ejecución → Ejecutar todas las celdas** (Ctrl + F9)
2. Celdas 1-3 crean la carpeta `Repositorio_Conocimiento/` con los 7 JSON
3. Celdas 4-9 definen las 6 clases POO del sistema
4. Celda 10 inicializa y verifica el sistema completo
5. Celda 11 ejecuta los **42 casos de prueba automáticos**
6. Celda 12 abre el **chat interactivo** — escribe `salir` para terminar

---
### Arquitectura del sistema (6 clases POO)
```
[Usuario]
    ↓ texto libre
[ChatbotEPIIS]  ← Fachada (Facade pattern) — única clase expuesta al notebook
    ├─ Preprocessor        lowercase · sin tildes · sin stopwords · tokenización
    ├─ KnowledgeBaseLoader carga 7 JSON · índices O(1) por intent_id
    ├─ IntentMatcher        fase 1: trigger phrases · fase 2: scoring ponderado
    ├─ ResponseBuilder      nivel 1: corpus · nivel 2: enriquecimiento normativo
    └─ ConversationHistory  registra turnos · genera reporte de evaluación
```


In [14]:
# ═══════════════════════════════════════════════════
# CELDA 1 — Imports y estructura de carpetas
# ═══════════════════════════════════════════════════
import os, json, re, unicodedata
from datetime import datetime
from pathlib import Path

BASE = Path("Repositorio_Conocimiento")
for d in ["corpus", "knowledge_base", "knowledge_files", "sources"]:
    (BASE / d).mkdir(parents=True, exist_ok=True)

print("✓ Estructura de carpetas creada")
for d in sorted(BASE.iterdir()):
    print(f"  {d}/")


✓ Estructura de carpetas creada
  Repositorio_Conocimiento/corpus/
  Repositorio_Conocimiento/knowledge_base/
  Repositorio_Conocimiento/knowledge_files/
  Repositorio_Conocimiento/sources/


In [15]:
# ═══════════════════════════════════════════════════
# CELDA 2 — Generar archivos JSON del repositorio
# ═══════════════════════════════════════════════════
import json
from pathlib import Path

BASE = Path("Repositorio_Conocimiento")

CORPUS_DATA = [
  {
    "id": "TUT-001",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Qué es la tutoría académica en la UNSAAC?",
    "respuesta": "La Tutoría Académica es un proceso permanente de acompañamiento durante la formación de los estudiantes, que se concreta mediante la atención personalizada o grupal por parte de docentes, orientándolos en los aspectos psicosociales, cognitivos y afectivos del aprendizaje (Art. 3°, Reglamento de Tutoría Académica UNSAAC, Res. N° CU-0220-2017-UNSAAC).",
    "intent": "consulta_tutoria_definicion",
    "variaciones": [
      "¿En qué consiste la tutoría?",
      "¿Cómo funciona la tutoría universitaria?",
      "¿Qué significa tutoría en la universidad?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 3°",
    "tags": [
      "tutoria",
      "acompanamiento",
      "definicion",
      "reglamento"
    ]
  },
  {
    "id": "TUT-002",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Con qué frecuencia debo asistir a las sesiones de tutoría?",
    "respuesta": "El reglamento establece un mínimo de tres encuentros por semestre: al inicio del curso académico, después de la primera evaluación parcial y una semana antes de la finalización del semestre (Art. 12.3). El tutor puede proponer sesiones adicionales según las necesidades del tutorado (Art. 12.4).",
    "intent": "consulta_tutoria_frecuencia",
    "variaciones": [
      "¿Cuántas veces al mes es la tutoría?",
      "¿Cada cuánto tiempo son las reuniones con el tutor?",
      "¿Qué tan seguido hay sesiones de tutoría?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 12.3 y 12.4",
    "tags": [
      "frecuencia",
      "sesiones",
      "semestre",
      "encuentros"
    ]
  },
  {
    "id": "TUT-003",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Quién es mi tutor asignado y cómo puedo contactarlo?",
    "respuesta": "El tutor es un docente con régimen de tiempo completo o dedicación exclusiva, asignado desde el inicio de los estudios hasta la culminación del plan curricular (Art. 14°). Puedes encontrar su nombre en el portal SERUNSA o en la secretaría de la escuela. El contacto se realiza vía correo institucional o en horario de atención publicado.",
    "intent": "consulta_tutoria_contacto_tutor",
    "variaciones": [
      "¿Cómo sé quién es mi tutor?",
      "¿Dónde encuentro el nombre de mi tutor?",
      "¿Cómo contacto a mi tutor académico?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 14°",
    "tags": [
      "tutor",
      "asignado",
      "contacto",
      "serunsa"
    ]
  },
  {
    "id": "TUT-004",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Qué temas se tratan en las sesiones de tutoría?",
    "respuesta": "Las sesiones abordan tres dimensiones: (a) académica: habilidades de estudio, estilos de aprendizaje, toma de decisiones; (b) personal: responsabilidad, habilidades interpersonales; y (c) profesional: orientación vocacional, proyección laboral (Art. 7°, Reglamento de Tutoría Académica UNSAAC, 2017).",
    "intent": "consulta_tutoria_temas",
    "variaciones": [
      "¿Sobre qué se habla en la tutoría?",
      "¿De qué tratan las reuniones con el tutor?",
      "¿Qué contenidos aborda la tutoría?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 7°",
    "tags": [
      "temas",
      "dimensiones",
      "academica",
      "personal",
      "profesional"
    ]
  },
  {
    "id": "TUT-005",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿La asistencia a tutoría es obligatoria?",
    "respuesta": "Sí. El Comité Tutorial de la Escuela determina la frecuencia mínima de entrevistas obligatorias (Art. 12.2). Las inasistencias reiteradas pueden reportarse a la coordinación académica. Adicionalmente, la asignación de tutor es obligatoria para estudiantes con matrícula condicionada (Art. 4°).",
    "intent": "consulta_tutoria_obligatoriedad",
    "variaciones": [
      "¿Es opcional asistir a tutoría?",
      "¿Me pueden sancionar si no voy a tutoría?",
      "¿Puedo faltar a las sesiones de tutoría?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 12.2 y Art. 4°",
    "tags": [
      "obligatoria",
      "obligatorio",
      "asistencia",
      "sancion"
    ]
  },
  {
    "id": "TUT-006",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Cómo se registra la asistencia a las sesiones de tutoría?",
    "respuesta": "El tutor lleva un registro oficial de visitas y reuniones, haciendo constar los problemas planteados y cualquier circunstancia relevante del seguimiento estudiantil. La plataforma informática de apoyo recoge también una valoración global al final del curso (Art. 13.7).",
    "intent": "consulta_tutoria_registro_asistencia",
    "variaciones": [
      "¿Cómo se controla la asistencia en tutoría?",
      "¿Hay lista de asistencia en tutoría?",
      "¿Cómo saber si se registró mi asistencia?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 13.7",
    "tags": [
      "registro",
      "asistencia",
      "lista",
      "plataforma"
    ]
  },
  {
    "id": "TUT-007",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Puedo cambiar de tutor si tengo dificultades con el asignado?",
    "respuesta": "Sí. Cualquier estudiante puede solicitar, de forma razonada, al Comité Tutorial de la Escuela el cambio de Tutor Académico; la solicitud se resuelve previa audiencia del tutor por el Comité y, en última instancia, por el Vicerrectorado Académico (Art. 14°, numeral 3).",
    "intent": "consulta_tutoria_cambio_tutor",
    "variaciones": [
      "¿Es posible solicitar otro tutor?",
      "¿Qué hago si no me llevo bien con mi tutor?",
      "¿Puedo pedir que me cambien de tutor?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 14° numeral 3",
    "tags": [
      "cambio",
      "tutor",
      "solicitud",
      "comite"
    ]
  },
  {
    "id": "TUT-008",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Existe algún reglamento que regule el proceso de tutorías?",
    "respuesta": "Sí. El Reglamento de Tutoría Académica fue aprobado mediante Resolución N° CU-0220-2017-UNSAAC de fecha 24 de mayo de 2017, con base en la Ley Universitaria N° 30220 (Art. 87.5) y el Estatuto de la UNSAAC (Art. 195.5). Está disponible en la página web institucional.",
    "intent": "consulta_tutoria_reglamento",
    "variaciones": [
      "¿Hay un reglamento de tutorías?",
      "¿Dónde puedo leer las normas de tutoría?",
      "¿Qué documento regula las tutorías en la UNSAAC?"
    ],
    "fuente": "Resolución N° CU-0220-2017-UNSAAC",
    "tags": [
      "reglamento",
      "norma",
      "resolucion",
      "ley"
    ]
  },
  {
    "id": "TUT-009",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Qué debo llevar a una sesión de tutoría?",
    "respuesta": "Se recomienda llevar el carné universitario, horario de clases, boleta de notas actualizada y cualquier documento relacionado con situaciones académicas o personales que se desee tratar. El tutor puede solicitar documentos adicionales según el tema de la sesión (Art. 10°).",
    "intent": "consulta_tutoria_materiales_sesion",
    "variaciones": [
      "¿Qué necesito para la reunión con mi tutor?",
      "¿Tengo que preparar algo para la tutoría?",
      "¿Qué documentos piden en tutoría?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 10°",
    "tags": [
      "llevar",
      "documentos",
      "carne",
      "notas"
    ]
  },
  {
    "id": "TUT-010",
    "categoria": "TUT",
    "categoria_nombre": "Proceso de Tutorías",
    "pregunta": "¿Las sesiones de tutoría son confidenciales?",
    "respuesta": "Los Tutores Académicos tienen deber de confidencialidad respecto de la información recibida de los estudiantes (Art. 15.1). En casos graves de carácter personal o académico, el tutor solo informará a padres o representantes legales en circunstancias excepcionales, procurando que sea el propio estudiante quien tome la iniciativa (Art. 13.8).",
    "intent": "consulta_tutoria_confidencialidad",
    "variaciones": [
      "¿El tutor puede compartir lo que le digo?",
      "¿Es privada la información en tutoría?",
      "¿Qué tan confidencial es la tutoría?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 15.1 y 13.8",
    "tags": [
      "confidencial",
      "privado",
      "datos",
      "secreto"
    ]
  },
  {
    "id": "TTUT-001",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Cuáles son los tipos de tutoría que ofrece la escuela?",
    "respuesta": "La tutoría académica en la UNSAAC se realiza de manera personalizada y grupal (Art. 7°), comprendiendo: (1) Tutoría Individual, (2) Tutoría Grupal y (3) Tutoría entre Pares. Cada modalidad atiende necesidades específicas del estudiante en distintas etapas de su formación.",
    "intent": "consulta_tipos_tutoria_general",
    "variaciones": [
      "¿Qué modalidades de tutoría existen?",
      "¿Cuántas clases de tutoría hay?",
      "¿De qué tipos es la tutoría en la UNSAAC?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 7°",
    "tags": [
      "tipos",
      "modalidades",
      "individual",
      "grupal",
      "pares"
    ]
  },
  {
    "id": "TTUT-002",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿En qué consiste la tutoría individual?",
    "respuesta": "La tutoría individual es una sesión privada entre el estudiante y su tutor asignado. Se activa especialmente cuando el estudiante presenta dificultades académicas, personales o de adaptación que requieren atención personalizada y seguimiento continuo por parte del docente tutor.",
    "intent": "consulta_tipos_tutoria_individual",
    "variaciones": [
      "¿Qué es la tutoría individual?",
      "¿Cómo funciona la tutoría uno a uno?",
      "¿Cuándo se da la tutoría personalizada?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 7°",
    "tags": [
      "individual",
      "personalizada",
      "privada",
      "uno a uno"
    ]
  },
  {
    "id": "TTUT-003",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Qué es la tutoría grupal y cómo se organiza?",
    "respuesta": "La tutoría grupal reúne a un conjunto de estudiantes del mismo semestre con un tutor. El número de tutorados por docente no debe superar los veinticinco (25) estudiantes en la medida de lo posible (Art. 14°, numeral 2), abordando temáticas comunes como rendimiento académico y orientación vocacional.",
    "intent": "consulta_tipos_tutoria_grupal",
    "variaciones": [
      "¿Cómo son las tutorías grupales?",
      "¿Cuántas personas van a la tutoría grupal?",
      "¿Con quiénes comparto la tutoría grupal?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 14° numeral 2",
    "tags": [
      "grupal",
      "grupo",
      "semestre",
      "25 estudiantes"
    ]
  },
  {
    "id": "TTUT-004",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Qué es la tutoría entre pares?",
    "respuesta": "La tutoría entre pares es una modalidad en la que estudiantes de semestres avanzados orientan a compañeros de semestres iniciales en temas académicos y de adaptación universitaria. Es voluntaria y complementa la tutoría docente. Los tutores pares actúan bajo supervisión de un docente ordinario.",
    "intent": "consulta_tipos_tutoria_pares",
    "variaciones": [
      "¿Qué es el tutoreo entre compañeros?",
      "¿Cómo funciona la tutoría entre estudiantes?",
      "¿Quiénes son los tutores pares?"
    ],
    "fuente": "Coordinación de Tutoría EPIIS",
    "tags": [
      "pares",
      "companeros",
      "avanzados",
      "voluntaria"
    ]
  },
  {
    "id": "TTUT-005",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Existe tutoría en línea o virtual?",
    "respuesta": "Sí, la tutoría virtual mediante plataformas como Zoom y Google Meet complementa la presencial. El tutor puede coordinar sesiones virtuales según la disponibilidad y necesidad del estudiante, especialmente cuando la presencialidad resulte difícil (en consonancia con Art. 12.1 del Reglamento de Tutoría).",
    "intent": "consulta_tipos_tutoria_virtual",
    "variaciones": [
      "¿Hay tutoría a distancia?",
      "¿Puedo hacer tutoría por videoconferencia?",
      "¿Las tutorías son presenciales o también virtuales?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 12.1",
    "tags": [
      "virtual",
      "online",
      "zoom",
      "distancia",
      "videoconferencia"
    ]
  },
  {
    "id": "TTUT-006",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Cuál es la diferencia entre tutoría académica y orientación psicológica?",
    "respuesta": "La tutoría académica acompaña el proceso formativo integral del estudiante. La orientación psicológica es un servicio especializado de la Unidad de Bienestar. El tutor puede derivar al estudiante a dicho servicio cuando detecta necesidades que superan su ámbito de acción (Art. 13.9 y Art. 10°).",
    "intent": "consulta_tipos_tutoria_vs_psicologia",
    "variaciones": [
      "¿La tutoría incluye apoyo psicológico?",
      "¿El tutor es psicólogo?",
      "¿Cuándo debo ir al psicólogo en vez de al tutor?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 13.9",
    "tags": [
      "psicologo",
      "psicologia",
      "diferencia",
      "bienestar"
    ]
  },
  {
    "id": "TTUT-007",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Qué es la tutoría de ingresantes?",
    "respuesta": "En el primer semestre de Estudios Generales, el Tutor Académico dedicará especial atención a facilitar la transición y adaptación de los estudiantes a la enseñanza universitaria (Art. 13.5). Esta modalidad busca prevenir el abandono temprano mediante acompañamiento reforzado.",
    "intent": "consulta_tipos_tutoria_ingresantes",
    "variaciones": [
      "¿Hay tutoría especial para los nuevos alumnos?",
      "¿Qué tutoría reciben los que recién entran?",
      "¿Cómo se atiende a los alumnos de primer semestre?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 13.5",
    "tags": [
      "ingresantes",
      "primer semestre",
      "adaptacion",
      "nuevos"
    ]
  },
  {
    "id": "TTUT-008",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Existe tutoría para estudiantes con bajo rendimiento?",
    "respuesta": "Sí. La asignación de tutor es obligatoria para el estudiante con matrícula condicionada (Art. 4°). El tutor implementa estrategias de atención reforzada y verifica la mejora del rendimiento académico hasta la finalización de los estudios (Art. 10°, numerales 2 y 3).",
    "intent": "consulta_tipos_tutoria_riesgo_academico",
    "variaciones": [
      "¿Hay tutoría especial si mis notas son bajas?",
      "¿Qué pasa si estoy en riesgo académico?",
      "¿Me asignan tutoría si jalo muchos cursos?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 4° y 10°",
    "tags": [
      "bajo rendimiento",
      "notas bajas",
      "riesgo academico",
      "condicionada"
    ]
  },
  {
    "id": "TTUT-009",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿Qué diferencia hay entre tutoría y asesoría de tesis?",
    "respuesta": "Son procesos distintos. La tutoría académica acompaña el proceso formativo general del estudiante. En los últimos cursos del pregrado se orienta hacia la preparación profesional (Art. 13.6), pero la asesoría de tesis es un proceso específico a cargo de un docente asesor designado por resolución del Director de Escuela.",
    "intent": "consulta_tipos_tutoria_vs_asesoria_tesis",
    "variaciones": [
      "¿Mi asesor de tesis es mi tutor?",
      "¿La tutoría incluye apoyo en la tesis?",
      "¿El tutor me ayuda con la tesis?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 13.6",
    "tags": [
      "tesis",
      "asesoria",
      "diferencia",
      "asesor"
    ]
  },
  {
    "id": "TTUT-010",
    "categoria": "TTUT",
    "categoria_nombre": "Tipos de Tutoría",
    "pregunta": "¿La tutoría cubre también orientación vocacional y de carrera?",
    "respuesta": "Sí. En los últimos cursos del pregrado, la actividad tutorial debe centrarse en la preparación del tutorado ante la futura salida a la vida profesional, en colaboración con los órganos y servicios establecidos para tal efecto por la Universidad (Art. 13.6).",
    "intent": "consulta_tipos_tutoria_vocacional",
    "variaciones": [
      "¿El tutor me ayuda a saber qué especialidad elegir?",
      "¿Puedo hablar con mi tutor sobre mi futuro profesional?",
      "¿La tutoría incluye orientación laboral?"
    ],
    "fuente": "Reglamento de Tutoría Académica UNSAAC, Art. 13.6",
    "tags": [
      "vocacional",
      "carrera",
      "profesional",
      "laboral"
    ]
  },
  {
    "id": "CUR-001",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cuántos semestres tiene la carrera de Ingeniería Informática y de Sistemas?",
    "respuesta": "La carrera de Ingeniería Informática y de Sistemas de la UNSAAC tiene una duración de 10 semestres académicos, equivalentes a 5 años de estudios. El plan de estudios vigente corresponde al currículo 2022.",
    "intent": "consulta_cursos_duracion_carrera",
    "variaciones": [
      "¿Cuántos ciclos tiene la carrera?",
      "¿Cuántos años dura la carrera?",
      "¿Qué duración tiene la carrera de sistemas?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "semestres",
      "duracion",
      "anos",
      "ciclos",
      "10 semestres"
    ]
  },
  {
    "id": "CUR-002",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cuántos créditos son necesarios para culminar la carrera?",
    "respuesta": "El plan de estudios 2022 de la EPIIS establece un total de 220 créditos académicos distribuidos en los 10 semestres, incluyendo cursos obligatorios, electivos, prácticas pre-profesionales y trabajo de titulación.",
    "intent": "consulta_cursos_total_creditos",
    "variaciones": [
      "¿Cuántos créditos necesito para acabar la carrera?",
      "¿Cuál es el total de créditos del plan de estudios?",
      "¿Cuántos créditos tiene el currículo?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "creditos",
      "total",
      "220",
      "plan de estudios"
    ]
  },
  {
    "id": "CUR-003",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cuáles son los cursos del primer semestre?",
    "respuesta": "El primer semestre incluye: Matemática Básica I, Introducción a la Ingeniería de Sistemas, Algorítmica y Programación I, Comunicación Oral y Escrita, Lógica y Filosofía, y Actividades Integradoras. La carga es de aproximadamente 22 créditos.",
    "intent": "consulta_cursos_primer_semestre",
    "variaciones": [
      "¿Qué cursos llevo en el primer ciclo?",
      "¿Cuáles son las materias de inicio de carrera?",
      "¿Qué se estudia en el primer semestre de sistemas?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "primer semestre",
      "cursos",
      "materias",
      "matematica",
      "programacion"
    ]
  },
  {
    "id": "CUR-004",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Puedo llevar cursos de otro semestre si aprobé los prerrequisitos?",
    "respuesta": "Sí, siempre que hayas aprobado los prerrequisitos establecidos en el plan de estudios y cuentes con autorización de la Dirección de la Escuela. La carga máxima permitida es de 24 créditos por semestre, salvo casos excepcionales aprobados por la Dirección.",
    "intent": "consulta_cursos_adelanto_semestres",
    "variaciones": [
      "¿Puedo adelantar cursos?",
      "¿Es posible llevar materias de ciclos superiores?",
      "¿Puedo llevar cursos de semestres más avanzados?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022 / Reglamento Académico UNSAAC",
    "tags": [
      "adelantar",
      "prerrequisitos",
      "avanzados",
      "24 creditos"
    ]
  },
  {
    "id": "CUR-005",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Qué pasa si desapruebo un curso?",
    "respuesta": "Si desapruebas un curso, debes matricularte nuevamente en él el siguiente semestre que sea ofertado. Un estudiante puede llevar el mismo curso un máximo de tres veces. A la tercera desaprobación, debe tramitar una solicitud especial ante el Consejo de Facultad.",
    "intent": "consulta_cursos_desaprobacion",
    "variaciones": [
      "¿Qué consecuencias tiene jalar un curso?",
      "¿Cuántas veces puedo llevar un curso?",
      "¿Qué hago si repruebo una materia?"
    ],
    "fuente": "Reglamento Académico UNSAAC",
    "tags": [
      "desaprobacion",
      "jalar",
      "reprobar",
      "tres veces",
      "consejo"
    ]
  },
  {
    "id": "CUR-006",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cómo está organizado el plan de estudios por áreas?",
    "respuesta": "El plan de estudios se organiza en cuatro áreas: (1) Formación General (humanidades, ciencias básicas), (2) Formación Profesional Básica (fundamentos de ingeniería), (3) Formación Profesional Específica (especialidad) y (4) Formación Complementaria (electivos, prácticas, titulación).",
    "intent": "consulta_cursos_areas_plan_estudios",
    "variaciones": [
      "¿Qué áreas tiene el currículo de la carrera?",
      "¿En qué grupos se dividen los cursos?",
      "¿Cómo se clasifican las materias de la carrera?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "areas",
      "plan de estudios",
      "formacion",
      "organizacion"
    ]
  },
  {
    "id": "CUR-007",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cuántos cursos electivos debo llevar durante la carrera?",
    "respuesta": "El plan de estudios 2022 exige un mínimo de 18 créditos en cursos electivos, que el estudiante puede elegir de entre las asignaturas ofertadas por la escuela o por otras unidades académicas, sujeto a disponibilidad y aprobación del Director de Escuela.",
    "intent": "consulta_cursos_electivos",
    "variaciones": [
      "¿Cuántas materias optativas son obligatorias?",
      "¿Cuántos electivos exige el plan de estudios?",
      "¿Puedo elegir mis cursos libremente?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "electivos",
      "optativas",
      "18 creditos",
      "libre eleccion"
    ]
  },
  {
    "id": "CUR-008",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Dónde puedo ver el horario de clases del semestre?",
    "respuesta": "Los horarios de clases están disponibles en el portal SERUNSA (serunsa.unsaac.edu.pe) en la sección 'Horarios'. También se publican en la página web de la escuela y en los tablones de anuncios al inicio de cada semestre académico.",
    "intent": "consulta_cursos_horario_clases",
    "variaciones": [
      "¿Cómo consulto mis horarios?",
      "¿Dónde están los horarios del semestre?",
      "¿En qué plataforma veo mis clases y aulas?"
    ],
    "fuente": "Portal SERUNSA - serunsa.unsaac.edu.pe",
    "tags": [
      "horario",
      "serunsa",
      "clases",
      "semestre"
    ]
  },
  {
    "id": "CUR-009",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Qué son los créditos académicos y cómo se calculan?",
    "respuesta": "Un crédito académico equivale a una hora de clase teórica semanal o dos horas de práctica o laboratorio semanal durante el semestre. El valor en créditos de cada asignatura está indicado en el plan de estudios y en el sílabo del curso.",
    "intent": "consulta_cursos_creditos_definicion",
    "variaciones": [
      "¿Qué significa un crédito académico?",
      "¿Cómo funciona el sistema de créditos?",
      "¿Un crédito equivale a cuántas horas?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022 / Reglamento Académico UNSAAC",
    "tags": [
      "creditos",
      "definicion",
      "horas",
      "calculo"
    ]
  },
  {
    "id": "CUR-010",
    "categoria": "CUR",
    "categoria_nombre": "Cursos y Semestres",
    "pregunta": "¿Cómo se calcula el promedio ponderado semestral?",
    "respuesta": "El promedio ponderado semestral se calcula multiplicando la nota final de cada curso por el número de créditos del mismo, sumando todos los productos y dividiendo entre el total de créditos matriculados. La escala de calificación es vigesimal (0–20).",
    "intent": "consulta_cursos_promedio_ponderado",
    "variaciones": [
      "¿Cómo se saca el promedio del semestre?",
      "¿Qué es el promedio ponderado?",
      "¿Cómo calcula el sistema mi promedio?"
    ],
    "fuente": "Reglamento Académico UNSAAC",
    "tags": [
      "promedio",
      "ponderado",
      "calculo",
      "notas",
      "vigesimal"
    ]
  },
  {
    "id": "ESP-001",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Cuáles son las especialidades que ofrece la Escuela de Ingeniería Informática y de Sistemas?",
    "respuesta": "La EPIIS ofrece cuatro líneas de especialización en los semestres superiores: (1) Ingeniería de Software, (2) Redes y Telecomunicaciones, (3) Inteligencia Artificial y Ciencia de Datos, y (4) Seguridad Informática. La elección se realiza a partir del VII semestre.",
    "intent": "consulta_esp_listado_especialidades",
    "variaciones": [
      "¿Qué especializaciones hay en la carrera?",
      "¿En qué áreas me puedo especializar?",
      "¿Cuántas especialidades tiene la carrera de sistemas?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "especialidades",
      "lineas",
      "software",
      "redes",
      "ia",
      "seguridad"
    ]
  },
  {
    "id": "ESP-002",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Cuándo debo elegir mi especialidad?",
    "respuesta": "La elección de especialidad se realiza al término del VI semestre, mediante un proceso de selección que considera el promedio ponderado acumulado del estudiante. La inscripción formal se realiza durante el proceso de matrícula del VII semestre.",
    "intent": "consulta_esp_cuando_elegir",
    "variaciones": [
      "¿En qué semestre escojo especialidad?",
      "¿A partir de qué ciclo elijo especialización?",
      "¿Cuándo me especializo en la carrera?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "elegir",
      "especialidad",
      "vi semestre",
      "vii semestre",
      "cuando"
    ]
  },
  {
    "id": "ESP-003",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Qué especialidad tiene más demanda laboral en la región?",
    "respuesta": "Según datos de la Bolsa de Trabajo UNSAAC y estudios de empleabilidad regional, las especialidades con mayor demanda en Cusco y la región sur del Perú son Ingeniería de Software e Inteligencia Artificial y Ciencia de Datos, seguidas por Seguridad Informática ante el creciente desarrollo del sector gubernamental y privado.",
    "intent": "consulta_esp_demanda_laboral",
    "variaciones": [
      "¿Cuál especialidad tiene más trabajo?",
      "¿Qué especialización es más solicitada en el mercado?",
      "¿Qué especialidad conviene más elegir?"
    ],
    "fuente": "Bolsa de Trabajo UNSAAC / Dirección de la Escuela Profesional",
    "tags": [
      "demanda",
      "laboral",
      "mercado",
      "trabajo",
      "empleabilidad"
    ]
  },
  {
    "id": "ESP-004",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Cuáles son los cursos de la especialidad de Ingeniería de Software?",
    "respuesta": "La especialidad de Ingeniería de Software incluye: Arquitectura de Software, Metodologías Ágiles, Diseño de Patrones, Ingeniería de Requisitos Avanzada, Pruebas y Calidad de Software, Gestión de Proyectos de TI y Desarrollo de Aplicaciones Móviles Avanzado.",
    "intent": "consulta_esp_cursos_ingenieria_software",
    "variaciones": [
      "¿Qué materias llevo en software?",
      "¿Qué cursos tiene la especialidad de software?",
      "¿Qué se aprende en Ingeniería de Software?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "software",
      "ingenieria de software",
      "cursos",
      "arquitectura",
      "agil"
    ]
  },
  {
    "id": "ESP-005",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Puedo cambiar de especialidad una vez que la elegí?",
    "respuesta": "El cambio de especialidad es posible mediante solicitud escrita dirigida al Director de la Escuela, con justificación fundamentada. El cambio está sujeto a disponibilidad de vacantes en la especialidad destino y se evalúa caso por caso. Solo se permite un cambio durante toda la carrera.",
    "intent": "consulta_esp_cambio_especialidad",
    "variaciones": [
      "¿Es posible cambiar de especialización?",
      "¿Qué pasa si quiero otra especialidad?",
      "¿Puedo arrepentirme de la especialidad que elegí?"
    ],
    "fuente": "Dirección de la Escuela Profesional EPIIS",
    "tags": [
      "cambio",
      "especialidad",
      "solicitud",
      "vacantes"
    ]
  },
  {
    "id": "ESP-006",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Qué cursos incluye la especialidad de Redes y Telecomunicaciones?",
    "respuesta": "La especialidad de Redes y Telecomunicaciones incluye: Redes Avanzadas, Protocolos de Comunicación, Administración de Redes, Redes Inalámbricas, VoIP y Telefonía IP, Virtualización de Redes (SDN), Ciberseguridad en Redes y Diseño de Infraestructuras de Red.",
    "intent": "consulta_esp_cursos_redes_telecom",
    "variaciones": [
      "¿Qué materias son de redes?",
      "¿Qué se estudia en telecomunicaciones?",
      "¿Qué aprendo en la especialidad de redes?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "redes",
      "telecomunicaciones",
      "cursos",
      "voip",
      "sdn"
    ]
  },
  {
    "id": "ESP-007",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Qué es la especialidad de Inteligencia Artificial y Ciencia de Datos?",
    "respuesta": "Esta especialidad forma profesionales capaces de diseñar, implementar y aplicar algoritmos de aprendizaje automático, minería de datos, procesamiento de lenguaje natural y análisis estadístico avanzado para la solución de problemas reales en organizaciones públicas y privadas.",
    "intent": "consulta_esp_ia_ciencia_datos",
    "variaciones": [
      "¿Qué es la especialidad de IA?",
      "¿Qué cubro en ciencia de datos?",
      "¿Qué se estudia en la especialidad de machine learning?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "inteligencia artificial",
      "ciencia de datos",
      "machine learning",
      "nlp",
      "mineria"
    ]
  },
  {
    "id": "ESP-008",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿La especialidad de Seguridad Informática tiene certificaciones asociadas?",
    "respuesta": "Sí, el programa de la especialidad de Seguridad Informática está alineado con estándares internacionales como CompTIA Security+, CEH (Certified Ethical Hacker) y CISSP. La escuela ofrece talleres preparatorios como complemento curricular para facilitar la obtención de estas certificaciones.",
    "intent": "consulta_esp_seguridad_certificaciones",
    "variaciones": [
      "¿En seguridad informática hay certificados?",
      "¿La especialidad de ciberseguridad prepara para certificaciones?",
      "¿Puedo obtener certificados en la especialidad de seguridad?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022 / Dirección de la Escuela Profesional",
    "tags": [
      "seguridad",
      "certificaciones",
      "ceh",
      "cissp",
      "ciberseguridad"
    ]
  },
  {
    "id": "ESP-009",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Las especialidades tienen convenios con empresas para prácticas?",
    "respuesta": "La EPIIS mantiene convenios con empresas del sector TI como SUNAT, BCP, empresas mineras y startups tecnológicas de la región, que priorizan a estudiantes de especialidades afines a sus requerimientos. El listado de empresas convenio está disponible en la Unidad de Prácticas de la escuela.",
    "intent": "consulta_esp_convenios_practicas",
    "variaciones": [
      "¿Las especialidades tienen empresas colaboradoras?",
      "¿Hay convenios de prácticas por especialidad?",
      "¿Las empresas contratan según mi especialidad?"
    ],
    "fuente": "Unidad de Prácticas EPIIS",
    "tags": [
      "convenios",
      "empresas",
      "practicas",
      "sunat",
      "bcp"
    ]
  },
  {
    "id": "ESP-010",
    "categoria": "ESP",
    "categoria_nombre": "Especialidades de la Escuela",
    "pregunta": "¿Puedo llevar cursos de otras especialidades como electivos?",
    "respuesta": "Sí, los cursos de especialidad de otras líneas pueden ser tomados como electivos, siempre que no haya cruce de horarios y se cuente con los prerrequisitos correspondientes. Esta opción permite ampliar el perfil profesional de manera interdisciplinaria.",
    "intent": "consulta_esp_electivos_cruzados",
    "variaciones": [
      "¿Puedo tomar materias de otra especialidad?",
      "¿Puedo cursar electivos de otra línea?",
      "¿Es posible mezclar cursos de diferentes especialidades?"
    ],
    "fuente": "Plan de Estudios EPIIS 2022",
    "tags": [
      "electivos",
      "cruzados",
      "interdisciplinario",
      "otra especialidad"
    ]
  },
  {
    "id": "PPP-001",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Cuáles son los requisitos para iniciar las prácticas pre-profesionales?",
    "respuesta": "Para iniciar las prácticas debes: (1) haber cursado y aprobado el IX semestre, (2) tener un promedio ponderado mínimo de 11, (3) contar con carta de presentación de la escuela, (4) tener acuerdo firmado con una empresa o institución convenio y (5) estar matriculado en el curso de Prácticas Pre-Profesionales.",
    "intent": "consulta_ppp_requisitos_inicio",
    "variaciones": [
      "¿Qué necesito para hacer prácticas?",
      "¿Cuáles son las condiciones para empezar las PPP?",
      "¿Qué debo tener para poder practicar?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "requisitos",
      "practicas",
      "ppp",
      "ix semestre",
      "promedio"
    ]
  },
  {
    "id": "PPP-002",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Cuántas horas de prácticas debo acumular?",
    "respuesta": "Las prácticas pre-profesionales tienen una duración mínima de 300 horas efectivas de trabajo en la empresa o institución, equivalentes a aproximadamente tres meses a tiempo completo. Este mínimo es exigido para la validación académica y para los trámites de titulación.",
    "intent": "consulta_ppp_horas_acumuladas",
    "variaciones": [
      "¿Cuántas horas son las prácticas?",
      "¿Qué duración tienen las prácticas pre-profesionales?",
      "¿Cuánto tiempo debo practicar?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "horas",
      "300 horas",
      "duracion",
      "tres meses"
    ]
  },
  {
    "id": "PPP-003",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿La empresa donde hago prácticas debe tener convenio con la UNSAAC?",
    "respuesta": "Se priorizan las empresas con convenio vigente con la UNSAAC. Sin embargo, es posible realizar prácticas en empresas sin convenio previo mediante carta de autorización de la Dirección de la Escuela, siempre que la institución esté formalmente constituida y las actividades correspondan al perfil de la carrera.",
    "intent": "consulta_ppp_empresa_convenio",
    "variaciones": [
      "¿Debo practicar en una empresa convenio?",
      "¿Puedo hacer prácticas en cualquier empresa?",
      "¿La empresa de prácticas necesita estar registrada en la universidad?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "empresa",
      "convenio",
      "autorizacion",
      "carta"
    ]
  },
  {
    "id": "PPP-004",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Cómo consigo la carta de presentación para las prácticas?",
    "respuesta": "La carta de presentación es emitida por la Dirección de la Escuela Profesional. Debes presentar: solicitud en la secretaría de la escuela, comprobante de matrícula vigente, constancia de notas actualizada y datos de la empresa receptora. El plazo de emisión es de tres días hábiles.",
    "intent": "consulta_ppp_carta_presentacion",
    "variaciones": [
      "¿Dónde tramito la carta de prácticas?",
      "¿Quién emite la carta de presentación para PPP?",
      "¿Cómo pido la carta para ir a practicar?"
    ],
    "fuente": "Dirección de la Escuela Profesional EPIIS",
    "tags": [
      "carta",
      "presentacion",
      "secretaria",
      "tres dias"
    ]
  },
  {
    "id": "PPP-005",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Qué es el informe de prácticas y cuándo debo presentarlo?",
    "respuesta": "El informe de prácticas es un documento técnico que describe las actividades realizadas, los aprendizajes obtenidos y los aportes a la organización. Debe presentarse al finalizar las prácticas ante la Unidad de Prácticas de la escuela, siguiendo el formato oficial disponible en la secretaría.",
    "intent": "consulta_ppp_informe_final",
    "variaciones": [
      "¿Tengo que escribir un informe de prácticas?",
      "¿Cómo es el informe final de PPP?",
      "¿Cuándo entrego el informe de mis prácticas?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "informe",
      "reporte",
      "formato",
      "unidad de practicas"
    ]
  },
  {
    "id": "PPP-006",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Las prácticas pre-profesionales son remuneradas?",
    "respuesta": "Según la Ley N° 28518 (Ley de Modalidades Formativas Laborales), el practicante debe recibir una subvención económica mensual no menor al 50% de la Remuneración Mínima Vital vigente, además de contar con seguro de salud. Estas condiciones deben estar estipuladas en el convenio de prácticas.",
    "intent": "consulta_ppp_remuneracion",
    "variaciones": [
      "¿Me pagan en las prácticas?",
      "¿Las PPP son con sueldo?",
      "¿Las empresas pagan a los practicantes?"
    ],
    "fuente": "Ley N° 28518 - Ley de Modalidades Formativas Laborales",
    "tags": [
      "remuneracion",
      "sueldo",
      "pago",
      "rmv",
      "subvencion"
    ]
  },
  {
    "id": "PPP-007",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Puedo hacer mis prácticas en una empresa pública o institución del Estado?",
    "respuesta": "Sí. Las entidades públicas como municipalidades, gobiernos regionales, ministerios y empresas estatales pueden ser centros de prácticas. Estas instituciones se rigen por el Decreto Legislativo N° 1401, que regula el régimen de formación laboral en el sector público.",
    "intent": "consulta_ppp_sector_publico",
    "variaciones": [
      "¿Puedo practicar en entidades del gobierno?",
      "¿Las entidades públicas aceptan practicantes?",
      "¿Puedo hacer PPP en una municipalidad o ministerio?"
    ],
    "fuente": "Decreto Legislativo N° 1401",
    "tags": [
      "publico",
      "estado",
      "municipalidad",
      "gobierno",
      "ministerio"
    ]
  },
  {
    "id": "PPP-008",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Quién supervisa y evalúa mis prácticas desde la universidad?",
    "respuesta": "La supervisión académica la realiza un docente designado como Asesor de Prácticas por la Dirección de la Escuela. Dicho asesor realiza visitas o reuniones de seguimiento, evalúa el informe final y recibe la calificación del supervisor de la empresa para emitir la nota definitiva.",
    "intent": "consulta_ppp_supervision_universidad",
    "variaciones": [
      "¿Qué docente supervisa las prácticas?",
      "¿Hay alguien de la UNSAAC que revise mis prácticas?",
      "¿Cómo me califica la universidad en las prácticas?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "supervisa",
      "asesor",
      "practicas",
      "evaluacion",
      "docente"
    ]
  },
  {
    "id": "PPP-009",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Puedo hacer las prácticas en la misma empresa donde trabajo?",
    "respuesta": "Es posible realizar las prácticas en el centro de trabajo actual, siempre que las actividades correspondan al perfil de la carrera y se cuente con un supervisor técnico dentro de la organización. Debe tramitarse la autorización respectiva ante la Dirección de la Escuela.",
    "intent": "consulta_ppp_trabajo_actual",
    "variaciones": [
      "¿Puedo validar mi trabajo como prácticas?",
      "¿Mi empleo actual puede contar como PPP?",
      "¿Puedo hacer las PPP donde ya trabajo?"
    ],
    "fuente": "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "tags": [
      "trabajo",
      "empleo",
      "validar",
      "autorizacion",
      "donde trabajo"
    ]
  },
  {
    "id": "PPP-010",
    "categoria": "PPP",
    "categoria_nombre": "Prácticas Pre-Profesionales",
    "pregunta": "¿Las prácticas pre-profesionales son un requisito para titularse?",
    "respuesta": "Sí. La aprobación del curso de Prácticas Pre-Profesionales y la presentación del informe correspondiente son requisitos obligatorios para iniciar el trámite de titulación en la EPIIS. El certificado de prácticas debe adjuntarse al expediente de titulación.",
    "intent": "consulta_ppp_requisito_titulacion",
    "variaciones": [
      "¿Sin prácticas no puedo titularme?",
      "¿Las PPP son obligatorias para el título?",
      "¿Las prácticas son parte de los requisitos de titulación?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "requisito",
      "titulacion",
      "obligatorio",
      "expediente",
      "certificado"
    ]
  },
  {
    "id": "BIE-001",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Qué servicios de bienestar ofrece la UNSAAC a sus estudiantes?",
    "respuesta": "La Dirección de Bienestar Universitario ofrece: (1) Atención médica y odontológica, (2) Orientación psicológica, (3) Servicio social y asistencia económica, (4) Comedor universitario, (5) Residencia universitaria, (6) Actividades deportivas y culturales, y (7) Seguro estudiantil.",
    "intent": "consulta_bie_servicios_generales",
    "variaciones": [
      "¿Qué ayudas da la universidad a los alumnos?",
      "¿Cuáles son los beneficios de bienestar universitario?",
      "¿Qué programas de apoyo tiene la UNSAAC?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "bienestar",
      "servicios",
      "apoyo",
      "medico",
      "psicologo",
      "comedor"
    ]
  },
  {
    "id": "BIE-002",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Cómo puedo acceder a la atención psicológica gratuita?",
    "respuesta": "La atención psicológica gratuita está disponible en la Unidad de Psicología de la Dirección de Bienestar Universitario. Puedes solicitar cita de forma presencial en las oficinas de bienestar o mediante el correo institucional de la unidad. El servicio es completamente confidencial.",
    "intent": "consulta_bie_atencion_psicologica",
    "variaciones": [
      "¿Dónde hay psicólogo en la UNSAAC?",
      "¿Cómo saco cita con el psicólogo?",
      "¿La universidad tiene servicio de salud mental?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "psicologo",
      "salud mental",
      "cita",
      "gratuito",
      "bienestar"
    ]
  },
  {
    "id": "BIE-003",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Qué es la beca de alimentación y cómo la solicito?",
    "respuesta": "La beca de alimentación otorga acceso al comedor universitario de lunes a viernes. Para solicitarla debes presentar: solicitud dirigida al Director de Bienestar, constancia de matrícula, declaración jurada de ingresos familiares y recibos de servicios del domicilio. Se otorga por semestre y se renueva previo estudio socioeconómico.",
    "intent": "consulta_bie_beca_alimentacion",
    "variaciones": [
      "¿Cómo accedo al comedor universitario gratis?",
      "¿Hay beca de comida en la UNSAAC?",
      "¿Puedo comer gratis en el comedor de la universidad?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "beca",
      "alimentacion",
      "comedor",
      "declaracion jurada",
      "solicitud"
    ]
  },
  {
    "id": "BIE-004",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Existe apoyo económico para estudiantes en situación de vulnerabilidad?",
    "respuesta": "Sí. La Unidad de Servicio Social evalúa casos de estudiantes en situación socioeconómica vulnerable y gestiona acceso a becas, exoneraciones de tasas y apoyo extraordinario. La UNSAAC articula también con el Programa Beca 18 del MINEDU para postulantes que cumplan el perfil.",
    "intent": "consulta_bie_apoyo_economico",
    "variaciones": [
      "¿La universidad da apoyo económico?",
      "¿Hay becas o subvenciones para alumnos pobres?",
      "¿Qué ayuda económica ofrece la UNSAAC?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "apoyo economico",
      "beca 18",
      "exoneracion",
      "vulnerabilidad",
      "minedu"
    ]
  },
  {
    "id": "BIE-005",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Cómo funciona el seguro estudiantil de la UNSAAC?",
    "respuesta": "Todo estudiante matriculado cuenta con seguro de accidentes personales vigente durante el año académico, que cubre gastos de atención médica por accidentes ocurridos dentro o fuera del campus. Para acceder, debes presentarte al tópico universitario o al centro de salud con tu carné vigente.",
    "intent": "consulta_bie_seguro_estudiantil",
    "variaciones": [
      "¿Qué cubre el seguro universitario?",
      "¿Tengo seguro por ser estudiante de la UNSAAC?",
      "¿El seguro estudiantil cubre accidentes?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "seguro",
      "accidentes",
      "carne",
      "topico",
      "cobertura"
    ]
  },
  {
    "id": "BIE-006",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Qué deportes y actividades culturales puedo practicar en la universidad?",
    "respuesta": "La Oficina de Deportes y Actividades Culturales ofrece disciplinas como fútbol, básquet, vóley, atletismo, natación, artes marciales y ajedrez. En el ámbito cultural hay talleres de danza, teatro, música y artes plásticas. La inscripción es gratuita para estudiantes matriculados.",
    "intent": "consulta_bie_deportes_cultura",
    "variaciones": [
      "¿Hay deportes en la UNSAAC?",
      "¿Qué actividades extracurriculares hay?",
      "¿Puedo hacer deporte en la universidad?"
    ],
    "fuente": "Oficina de Deportes y Actividades Culturales UNSAAC",
    "tags": [
      "deportes",
      "cultura",
      "futbol",
      "basquet",
      "danza",
      "teatro",
      "gratuito"
    ]
  },
  {
    "id": "BIE-007",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿La UNSAAC tiene residencia universitaria para estudiantes del interior?",
    "respuesta": "Sí, la UNSAAC cuenta con residencia universitaria para estudiantes provenientes de provincias y comunidades alejadas. El acceso se determina mediante estudio socioeconómico. Para postular, debes presentar la solicitud al inicio de cada año académico en la Dirección de Bienestar Universitario.",
    "intent": "consulta_bie_residencia_universitaria",
    "variaciones": [
      "¿Hay albergue para estudiantes foráneos?",
      "¿La universidad tiene dormitorios para alumnos?",
      "¿Existe residencia universitaria en la UNSAAC?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "residencia",
      "albergue",
      "dormitorio",
      "foraneos",
      "provincias"
    ]
  },
  {
    "id": "BIE-008",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿Cómo solicito un certificado de situación socioeconómica?",
    "respuesta": "El certificado lo emite la Unidad de Servicio Social de Bienestar Universitario. Debes presentar: solicitud, constancia de matrícula, declaración jurada de ingresos familiares y documentos que acrediten las condiciones descritas. El proceso tarda entre 5 y 10 días hábiles.",
    "intent": "consulta_bie_certificado_socioeconomico",
    "variaciones": [
      "¿Dónde tramito la constancia socioeconómica?",
      "¿Cómo obtengo el estudio socioeconómico?",
      "¿La UNSAAC hace estudios socioeconómicos?"
    ],
    "fuente": "Unidad de Servicio Social UNSAAC",
    "tags": [
      "certificado",
      "socioeconomico",
      "constancia",
      "servicio social",
      "dias habiles"
    ]
  },
  {
    "id": "BIE-009",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿El comedor universitario está disponible para todos los estudiantes?",
    "respuesta": "El comedor universitario está abierto para todos los estudiantes matriculados. Los beneficiarios de beca de alimentación acceden sin costo. Los demás estudiantes pueden adquirir tickets al precio subsidiado establecido por la universidad, significativamente menor al del mercado.",
    "intent": "consulta_bie_comedor_acceso",
    "variaciones": [
      "¿Puedo comer en el comedor aunque no tenga beca?",
      "¿El comedor es abierto para todos?",
      "¿Cuánto cuesta el menú del comedor universitario?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "comedor",
      "precio",
      "subsidiado",
      "tickets",
      "todos"
    ]
  },
  {
    "id": "BIE-010",
    "categoria": "BIE",
    "categoria_nombre": "Bienestar Universitario",
    "pregunta": "¿A qué número o correo puedo comunicarme con Bienestar Universitario?",
    "respuesta": "La Dirección de Bienestar Universitario está ubicada en el Campus Universitario de Perayoc. Puedes contactarlos a través del correo institucional publicado en el portal web de la UNSAAC (unsaac.edu.pe) o llamando a la central telefónica de la universidad, anexo de Bienestar.",
    "intent": "consulta_bie_contacto_oficina",
    "variaciones": [
      "¿Cuál es el teléfono de bienestar universitario?",
      "¿Cómo contacto a la oficina de bienestar?",
      "¿Dónde está la oficina de bienestar universitario?"
    ],
    "fuente": "Portal Web UNSAAC",
    "tags": [
      "contacto",
      "correo",
      "telefono",
      "perayoc",
      "bienestar"
    ]
  },
  {
    "id": "MOV-001",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Qué programas de movilidad estudiantil ofrece la UNSAAC?",
    "respuesta": "La UNSAAC ofrece movilidad estudiantil a través de: (1) PAME-UDUAL (movilidad latinoamericana), (2) convenios bilaterales con universidades de España, Brasil, México y Colombia, (3) Programa de Movilidad Nacional con universidades públicas del Perú, y (4) becas de posgrado PRONABEC y Fundación Carolina.",
    "intent": "consulta_mov_programas_disponibles",
    "variaciones": [
      "¿Hay intercambios estudiantiles en la UNSAAC?",
      "¿Puedo estudiar en otra universidad con la UNSAAC?",
      "¿Qué convenios de intercambio tiene la universidad?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "movilidad",
      "intercambio",
      "pame",
      "udual",
      "convenios",
      "pronabec"
    ]
  },
  {
    "id": "MOV-002",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Cuáles son los requisitos mínimos para participar en un programa de intercambio?",
    "respuesta": "Los requisitos generales son: (1) estar matriculado a partir del IV semestre, (2) tener promedio ponderado mínimo de 14 sobre 20, (3) acreditar nivel intermedio del idioma del país destino, (4) no tener sanciones disciplinarias y (5) presentar carta de motivación y plan de estudios equivalente.",
    "intent": "consulta_mov_requisitos_intercambio",
    "variaciones": [
      "¿Qué se necesita para el intercambio estudiantil?",
      "¿Qué requisitos piden para la movilidad?",
      "¿Cómo postulo a un intercambio universitario?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "requisitos",
      "intercambio",
      "promedio 14",
      "idioma",
      "carta motivacion"
    ]
  },
  {
    "id": "MOV-003",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Los cursos llevados en el intercambio se convalidan en la UNSAAC?",
    "respuesta": "Sí. Los cursos aprobados durante la movilidad se pueden convalidar mediante un proceso de homologación ante la Dirección de la Escuela, previa revisión de sílabos y contenidos. El convenio establece equivalencias curriculares antes del viaje para garantizar el reconocimiento.",
    "intent": "consulta_mov_convalidacion_cursos",
    "variaciones": [
      "¿Me reconocen los cursos del intercambio?",
      "¿Puedo convalidar materias del intercambio?",
      "¿Los créditos del intercambio cuentan en mi carrera?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "convalidacion",
      "homologacion",
      "creditos",
      "silabos",
      "reconocimiento"
    ]
  },
  {
    "id": "MOV-004",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Hay apoyo económico para los estudiantes que van de intercambio?",
    "respuesta": "Algunos programas incluyen cobertura de manutención y/o pasajes. La universidad articula con PRONABEC, la Fundación Carolina y otros fondos para becas de movilidad. La Oficina de Relaciones Internacionales orienta a los postulantes sobre las fuentes de financiamiento disponibles por convocatoria.",
    "intent": "consulta_mov_apoyo_economico",
    "variaciones": [
      "¿Dan beca para el intercambio?",
      "¿La universidad financia el intercambio?",
      "¿Hay ayuda para pagar el intercambio estudiantil?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "apoyo",
      "beca",
      "financiamiento",
      "pronabec",
      "fundacion carolina"
    ]
  },
  {
    "id": "MOV-005",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Puedo recibir estudiantes de intercambio de otras universidades?",
    "respuesta": "Sí. La UNSAAC recibe estudiantes de intercambio entrante (incoming). Estos estudiantes se matriculan en cursos regulares de la carrera y reciben orientación de la Oficina de Relaciones Internacionales. El proceso de admisión se gestiona a través de los convenios bilaterales vigentes.",
    "intent": "consulta_mov_estudiantes_entrantes",
    "variaciones": [
      "¿La UNSAAC recibe estudiantes extranjeros?",
      "¿Hay estudiantes de intercambio que vienen a la UNSAAC?",
      "¿Cómo funciona la movilidad entrante?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "entrantes",
      "incoming",
      "extranjeros",
      "admision",
      "bilateral"
    ]
  },
  {
    "id": "MOV-006",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Dónde puedo enterarme de las convocatorias de movilidad estudiantil?",
    "respuesta": "Las convocatorias se publican en: la página web oficial de la UNSAAC (sección Relaciones Internacionales), el portal SERUNSA, las redes sociales institucionales y los paneles informativos de la facultad. Se recomienda estar atento al inicio de cada semestre académico.",
    "intent": "consulta_mov_convocatorias",
    "variaciones": [
      "¿Cómo me entero de las becas de intercambio?",
      "¿Dónde se publican las convocatorias de movilidad?",
      "¿Cómo saber cuándo hay intercambios disponibles?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "convocatorias",
      "publicacion",
      "serunsa",
      "redes sociales",
      "semestre"
    ]
  },
  {
    "id": "MOV-007",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Qué documentos necesito para postular a la movilidad estudiantil?",
    "respuesta": "El expediente generalmente incluye: solicitud de postulación, currículum vitae, constancia de matrícula, certificado de notas, carta de motivación, carta de recomendación de un docente, acreditación del idioma y el plan de estudios equivalente propuesto.",
    "intent": "consulta_mov_documentos_postulacion",
    "variaciones": [
      "¿Qué papeles piden para el intercambio?",
      "¿Qué expediente armo para la movilidad?",
      "¿Qué documentación necesito para postular al intercambio?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "documentos",
      "expediente",
      "curriculum",
      "constancia",
      "carta recomendacion"
    ]
  },
  {
    "id": "MOV-008",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿El intercambio afecta mi proceso de titulación o retrasa mi carrera?",
    "respuesta": "No necesariamente. La clave es planificar el intercambio con la Dirección de la Escuela y la Oficina de Relaciones Internacionales para garantizar la equivalencia curricular. Los créditos aprobados en la universidad destino se homologan, por lo que el intercambio puede realizarse sin retrasar la culminación de la carrera.",
    "intent": "consulta_mov_impacto_titulacion",
    "variaciones": [
      "¿El intercambio me retrasa?",
      "¿Ir de intercambio atrasa la titulación?",
      "¿Pierdo semestre si hago intercambio?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "retrasar",
      "titulacion",
      "homologacion",
      "creditos",
      "planificar"
    ]
  },
  {
    "id": "MOV-009",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿Necesito dominar otro idioma para participar en la movilidad?",
    "respuesta": "Depende del programa y el país destino. Para universidades latinoamericanas, el español es suficiente. Para destinos en Europa o EE.UU., se exige acreditar nivel intermedio o avanzado del idioma local (inglés, portugués o francés), generalmente mediante certificados como TOEFL, IELTS o equivalentes.",
    "intent": "consulta_mov_requisito_idioma",
    "variaciones": [
      "¿Es obligatorio hablar otro idioma para el intercambio?",
      "¿Qué nivel de inglés piden para la movilidad?",
      "¿Puedo ir a intercambio si no hablo inglés?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "idioma",
      "ingles",
      "toefl",
      "ielts",
      "nivel",
      "portugues"
    ]
  },
  {
    "id": "MOV-010",
    "categoria": "MOV",
    "categoria_nombre": "Movilidad Estudiantil",
    "pregunta": "¿La UNSAAC tiene convenios con universidades reconocidas internacionalmente?",
    "respuesta": "Sí. La UNSAAC mantiene convenios activos con universidades como la Universidad de Salamanca (España), la Universidad Federal de Paraná (Brasil), la UNAM (México), la Universidad Nacional de Córdoba (Argentina) y varias instituciones agrupadas en redes como UDUAL y AUIP.",
    "intent": "consulta_mov_universidades_convenio",
    "variaciones": [
      "¿Con qué universidades del mundo tiene convenio la UNSAAC?",
      "¿La UNSAAC tiene alianzas internacionales?",
      "¿Qué universidades extranjeras tienen acuerdo con la UNSAAC?"
    ],
    "fuente": "Oficina de Relaciones Internacionales UNSAAC",
    "tags": [
      "convenios",
      "salamanca",
      "unam",
      "brasil",
      "argentina",
      "udual"
    ]
  },
  {
    "id": "MAT-001",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Cuándo se realizan las matrículas en la UNSAAC?",
    "respuesta": "El proceso de matrícula se realiza al inicio de cada semestre académico, conforme al cronograma publicado por la DRAA. Generalmente se realiza en los meses de marzo (semestre impar) y agosto (semestre par). Las fechas exactas se publican en el portal SERUNSA.",
    "intent": "consulta_mat_fechas_matricula",
    "variaciones": [
      "¿En qué fechas es la matrícula?",
      "¿Cuándo empieza la matrícula?",
      "¿Cuáles son los periodos de matrícula?"
    ],
    "fuente": "Dirección de Registros Académicos y Admisión (DRAA)",
    "tags": [
      "matricula",
      "fechas",
      "cronograma",
      "serunsa",
      "marzo",
      "agosto"
    ]
  },
  {
    "id": "MAT-002",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Cómo me matriculo en el portal SERUNSA?",
    "respuesta": "Para matricularte en SERUNSA: (1) Ingresa a serunsa.unsaac.edu.pe con tu usuario y contraseña, (2) selecciona 'Proceso de Matrícula', (3) elige los cursos disponibles según tu semestre y especialidad, (4) verifica el horario y confirma la selección, (5) imprime o guarda el comprobante de matrícula.",
    "intent": "consulta_mat_proceso_serunsa",
    "variaciones": [
      "¿Cómo se hace la matrícula en línea?",
      "¿Cuáles son los pasos para matricularme?",
      "¿Cómo uso el sistema SERUNSA para matricularme?"
    ],
    "fuente": "Portal SERUNSA - serunsa.unsaac.edu.pe",
    "tags": [
      "serunsa",
      "matricula",
      "proceso",
      "pasos",
      "usuario",
      "contrasena"
    ]
  },
  {
    "id": "MAT-003",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Cuáles son los requisitos para poder matricularme?",
    "respuesta": "Los requisitos son: (1) no tener deudas económicas pendientes con la universidad, (2) tener el carné universitario vigente, (3) haber cumplido con los trámites administrativos del semestre anterior, (4) no estar sancionado académicamente y (5) contar con la habilitación del sistema SERUNSA.",
    "intent": "consulta_mat_requisitos",
    "variaciones": [
      "¿Qué necesito para hacer la matrícula?",
      "¿Qué condiciones debo cumplir para matricularme?",
      "¿Qué me pueden impedir matricularme?"
    ],
    "fuente": "Reglamento de Matrícula UNSAAC",
    "tags": [
      "requisitos",
      "deudas",
      "carne",
      "habilitacion",
      "serunsa"
    ]
  },
  {
    "id": "MAT-004",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Puedo hacer la matrícula de forma presencial?",
    "respuesta": "El proceso de matrícula es principalmente virtual a través del portal SERUNSA. Sin embargo, la DRAA habilita módulos de atención presencial para estudiantes con dificultades técnicas o situaciones especiales. Consulta el cronograma de atención publicado en la secretaría de la facultad.",
    "intent": "consulta_mat_modalidad_presencial",
    "variaciones": [
      "¿La matrícula se puede hacer en persona?",
      "¿Hay matrícula presencial en la UNSAAC?",
      "¿Dónde hago la matrícula si no tengo internet?"
    ],
    "fuente": "Dirección de Registros Académicos y Admisión (DRAA)",
    "tags": [
      "presencial",
      "virtual",
      "draa",
      "modulos",
      "secretaria"
    ]
  },
  {
    "id": "MAT-005",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Qué hago si no pude matricularme dentro del plazo establecido?",
    "respuesta": "Si no te matriculaste en el plazo regular, puedes solicitar la matrícula extemporánea mediante solicitud justificada ante el Director de la Escuela dentro de los primeros cinco días del inicio de clases. Esta modalidad está sujeta a evaluación y disponibilidad de vacantes, y puede implicar un pago adicional.",
    "intent": "consulta_mat_fuera_de_plazo",
    "variaciones": [
      "¿Qué pasa si se me venció el plazo de matrícula?",
      "¿Puedo hacer matrícula extemporánea?",
      "¿Me pueden matricular fuera de fecha?"
    ],
    "fuente": "Reglamento de Matrícula UNSAAC",
    "tags": [
      "extemporanea",
      "fuera de plazo",
      "solicitud",
      "cinco dias",
      "vacantes"
    ]
  },
  {
    "id": "MAT-006",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Cuántos créditos máximos puedo llevar por semestre?",
    "respuesta": "La carga máxima es de 24 créditos por semestre para el régimen regular. Los estudiantes con promedio ponderado acumulado igual o mayor a 15 pueden solicitar una sobrecarga de hasta 2 créditos adicionales. La carga mínima para mantener la condición de estudiante regular es de 12 créditos.",
    "intent": "consulta_mat_creditos_maximo",
    "variaciones": [
      "¿Cuál es la carga máxima de créditos?",
      "¿Cuántos cursos puedo llevar a la vez?",
      "¿Hay límite de materias por semestre?"
    ],
    "fuente": "Reglamento Académico UNSAAC",
    "tags": [
      "creditos",
      "maximo",
      "24 creditos",
      "sobrecarga",
      "12 creditos",
      "minimo"
    ]
  },
  {
    "id": "MAT-007",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Puedo retirar un curso después de matricularme?",
    "respuesta": "Sí. Puedes solicitar el retiro de uno o más cursos hasta la cuarta semana de inicio de clases, sin que quede registro académico negativo. Luego de ese plazo, el retiro solo procede por causas debidamente justificadas mediante solicitud al Director de Escuela.",
    "intent": "consulta_mat_retiro_curso",
    "variaciones": [
      "¿Puedo cancelar un curso en el que me matriculé?",
      "¿Es posible dar de baja un curso?",
      "¿Hasta cuándo puedo retirar un curso?"
    ],
    "fuente": "Reglamento Académico UNSAAC",
    "tags": [
      "retiro",
      "baja",
      "cancelar",
      "cuarta semana",
      "justificado"
    ]
  },
  {
    "id": "MAT-008",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Qué es la matrícula condicional y cuándo aplica?",
    "respuesta": "La matrícula condicional se aplica a estudiantes que han desaprobado el mismo curso en dos oportunidades, o cuyo promedio ponderado está por debajo del mínimo establecido. Implica un seguimiento académico reforzado y la firma de un acta de compromiso ante la Dirección de la Escuela.",
    "intent": "consulta_mat_condicional",
    "variaciones": [
      "¿En qué casos hay matrícula condicional?",
      "¿Qué es la matrícula con condición?",
      "¿Me pueden poner matrícula condicional?"
    ],
    "fuente": "Reglamento Académico UNSAAC",
    "tags": [
      "condicional",
      "bajo promedio",
      "seguimiento",
      "acta de compromiso",
      "dos veces"
    ]
  },
  {
    "id": "MAT-009",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Cuánto cuesta la matrícula en la UNSAAC?",
    "respuesta": "La UNSAAC es una universidad pública que cobra tasas de matrícula establecidas anualmente por el Consejo Universitario. Estas tasas varían según el nivel socioeconómico del estudiante, determinado por el estudio realizado por Bienestar Universitario. Los montos actualizados se publican en el portal SERUNSA al inicio del semestre.",
    "intent": "consulta_mat_costo",
    "variaciones": [
      "¿Cuánto es la tasa de matrícula?",
      "¿Qué costo tiene matricularse en la UNSAAC?",
      "¿Cuál es el precio de la matrícula universitaria?"
    ],
    "fuente": "Consejo Universitario UNSAAC / Portal SERUNSA",
    "tags": [
      "costo",
      "precio",
      "tasa",
      "socioeconomico",
      "serunsa"
    ]
  },
  {
    "id": "MAT-010",
    "categoria": "MAT",
    "categoria_nombre": "Matrícula",
    "pregunta": "¿Qué es el seguro universitario y cuándo se activa con la matrícula?",
    "respuesta": "Al completar la matrícula el seguro estudiantil de accidentes personales se activa automáticamente para el semestre en curso. La cobertura inicia desde el primer día de clases y finaliza al término del semestre académico. Para usarlo, debes presentar tu carné universitario vigente.",
    "intent": "consulta_mat_seguro_activacion",
    "variaciones": [
      "¿El seguro se activa solo al matricularme?",
      "¿Cuándo empieza a cubrirme el seguro estudiantil?",
      "¿La matrícula incluye seguro de accidentes?"
    ],
    "fuente": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
    "tags": [
      "seguro",
      "activacion",
      "automatico",
      "carne",
      "cobertura"
    ]
  },
  {
    "id": "TIT-001",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cuáles son las modalidades de titulación disponibles en la EPIIS?",
    "respuesta": "La EPIIS ofrece cuatro modalidades de titulación: (1) Tesis de investigación, (2) Trabajo de suficiencia profesional, (3) Trabajo de investigación (para bachilleres automáticos Ley N° 30220) y (4) Examen de competencia profesional. Cada modalidad tiene requisitos y plazos específicos definidos en el reglamento.",
    "intent": "consulta_tit_modalidades",
    "variaciones": [
      "¿Cómo me puedo titular?",
      "¿Qué opciones hay para obtener el título?",
      "¿Cuántas formas de titulación existen en la carrera?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "titulacion",
      "modalidades",
      "tesis",
      "suficiencia",
      "bachiller",
      "examen"
    ]
  },
  {
    "id": "TIT-002",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cuáles son los requisitos para iniciar el proceso de titulación?",
    "respuesta": "Los requisitos generales son: (1) haber aprobado todos los cursos del plan de estudios (220 créditos), (2) tener el grado de Bachiller, (3) haber aprobado las prácticas pre-profesionales, (4) no tener deudas con la universidad y (5) presentar el expediente completo a la Secretaría de Grados y Títulos de la Facultad.",
    "intent": "consulta_tit_requisitos_inicio",
    "variaciones": [
      "¿Qué necesito para titularme?",
      "¿Cuáles son los requisitos del proceso de titulación?",
      "¿Qué documentos piden para el título?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "requisitos",
      "titulacion",
      "220 creditos",
      "bachiller",
      "practicas",
      "expediente"
    ]
  },
  {
    "id": "TIT-003",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cómo obtengo el grado de Bachiller?",
    "respuesta": "Según la Ley Universitaria N° 30220, el grado de Bachiller se otorga automáticamente al haber aprobado todos los créditos del plan de estudios y el trabajo de investigación correspondiente. El trámite se realiza ante la Secretaría de Grados y Títulos presentando el expediente con los documentos requeridos.",
    "intent": "consulta_tit_grado_bachiller",
    "variaciones": [
      "¿Qué se necesita para el bachillerato?",
      "¿Cuáles son los requisitos para el grado de bachiller?",
      "¿El bachiller es automático?"
    ],
    "fuente": "Ley Universitaria N° 30220 / Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "bachiller",
      "automatico",
      "grado",
      "investigacion",
      "ley 30220"
    ]
  },
  {
    "id": "TIT-004",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cuánto tiempo tengo para titularme después de terminar los cursos?",
    "respuesta": "El Reglamento de Grados y Títulos no establece un plazo máximo para titularse tras culminar los estudios. Sin embargo, se recomienda iniciar el proceso a la brevedad para evitar posibles cambios en el reglamento o actualización del plan de estudios que pudieran afectar los requisitos.",
    "intent": "consulta_tit_plazo_titulacion",
    "variaciones": [
      "¿Hay un plazo límite para titularme?",
      "¿Cuánto tiempo tengo para sacar mi título?",
      "¿Me caducan los créditos si no me titulo?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "plazo",
      "titulacion",
      "limite",
      "caducidad",
      "creditos"
    ]
  },
  {
    "id": "TIT-005",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Qué es un trabajo de suficiencia profesional?",
    "respuesta": "El trabajo de suficiencia profesional es una modalidad de titulación en la que el egresado sustenta un informe detallado sobre una experiencia laboral o proyecto relevante desarrollado durante su desempeño profesional. Demuestra la aplicación de competencias propias de la carrera en un contexto real.",
    "intent": "consulta_tit_suficiencia_profesional",
    "variaciones": [
      "¿Qué es la suficiencia profesional?",
      "¿Cómo funciona la titulación por suficiencia?",
      "¿En qué consiste la modalidad de suficiencia profesional?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "suficiencia",
      "profesional",
      "experiencia laboral",
      "proyecto",
      "informe"
    ]
  },
  {
    "id": "TIT-006",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cómo escojo a mi asesor de tesis?",
    "respuesta": "Puedes proponer a un docente ordinario de la Escuela como asesor, sujeto a su disponibilidad y a que su perfil de investigación sea compatible con el tema. La designación oficial la realiza el Director de Escuela mediante resolución. Un asesor puede tener como máximo cinco tesistas simultáneos.",
    "intent": "consulta_tit_eleccion_asesor",
    "variaciones": [
      "¿Quién puede ser mi asesor de tesis?",
      "¿Cómo se asigna el asesor de investigación?",
      "¿Puedo elegir a mi asesor?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "asesor",
      "tesis",
      "docente",
      "cinco tesistas",
      "resolucion"
    ]
  },
  {
    "id": "TIT-007",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Cuánto tiempo toma habitualmente el proceso de titulación?",
    "respuesta": "El proceso varía según la modalidad. Por tesis, el tiempo promedio es de 12 a 24 meses desde la aprobación del proyecto hasta la sustentación. Por suficiencia profesional, puede resolverse en 3 a 6 meses. La emisión del título físico por la SUNEDU puede añadir 3 a 6 meses adicionales.",
    "intent": "consulta_tit_duracion_proceso",
    "variaciones": [
      "¿Cuánto dura titularse?",
      "¿En cuánto tiempo puedo sacar mi título?",
      "¿El trámite de titulación es rápido?"
    ],
    "fuente": "Secretaría de Grados y Títulos EPIIS",
    "tags": [
      "duracion",
      "tiempo",
      "meses",
      "tesis",
      "suficiencia",
      "sunedu"
    ]
  },
  {
    "id": "TIT-008",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿En qué consiste el jurado de sustentación?",
    "respuesta": "El jurado está conformado por tres docentes ordinarios de la escuela o la facultad: un presidente, un primer miembro y un segundo miembro. El asesor no forma parte del jurado calificador, pero puede estar presente. La nota de sustentación se calcula promediando las calificaciones de los tres jurados.",
    "intent": "consulta_tit_jurado_sustentacion",
    "variaciones": [
      "¿Quiénes conforman el jurado de tesis?",
      "¿Cómo es el jurado de la sustentación?",
      "¿Cuántas personas me evalúan en la sustentación?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "jurado",
      "tres docentes",
      "presidente",
      "sustentacion",
      "promedio"
    ]
  },
  {
    "id": "TIT-009",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Qué pasa si desapruebo la sustentación?",
    "respuesta": "Si desapruebas la sustentación, puedes solicitar una nueva fecha en un plazo no menor a treinta días calendario, previa revisión y corrección de las observaciones señaladas por el jurado. Se permiten hasta dos oportunidades de sustentación dentro del proceso de titulación.",
    "intent": "consulta_tit_desaprobacion_sustentacion",
    "variaciones": [
      "¿Puedo repetir la sustentación si la jalé?",
      "¿Qué ocurre si no apruebo la defensa de tesis?",
      "¿Tengo otra oportunidad si desapruebo la sustentación?"
    ],
    "fuente": "Reglamento de Grados y Títulos UNSAAC",
    "tags": [
      "desaprobacion",
      "sustentacion",
      "30 dias",
      "dos oportunidades",
      "observaciones"
    ]
  },
  {
    "id": "TIT-010",
    "categoria": "TIT",
    "categoria_nombre": "Titulación",
    "pregunta": "¿Dónde debo registrar mi título una vez obtenido?",
    "respuesta": "El título profesional es registrado automáticamente por la UNSAAC ante la SUNEDU (Superintendencia Nacional de Educación Superior Universitaria), que emite el código único de registro. Una vez inscrito en SUNEDU, el título tiene validez nacional. Puedes verificar su registro en el portal verificador de SUNEDU.",
    "intent": "consulta_tit_registro_sunedu",
    "variaciones": [
      "¿El título se inscribe en algún registro?",
      "¿Dónde se registra el título profesional?",
      "¿Qué hago después de obtener el título?"
    ],
    "fuente": "SUNEDU - Superintendencia Nacional de Educación Superior Universitaria",
    "tags": [
      "sunedu",
      "registro",
      "titulo",
      "validez",
      "nacional",
      "codigo"
    ]
  },
  {
    "id": "SER-001",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo accedo a los recursos de la biblioteca universitaria?",
    "respuesta": "La Biblioteca Central UNSAAC ofrece acceso presencial de lunes a sábado y acceso virtual a través del repositorio institucional (repositorio.unsaac.edu.pe), bases de datos suscritas (EBSCO, Scopus) y la Biblioteca Virtual del Consorcio de Universidades. El acceso digital requiere usuario institucional.",
    "intent": "consulta_ser_biblioteca",
    "variaciones": [
      "¿Puedo usar la biblioteca de la UNSAAC?",
      "¿Cómo uso la biblioteca universitaria?",
      "¿La biblioteca tiene acceso en línea?"
    ],
    "fuente": "Biblioteca Central UNSAAC",
    "tags": [
      "biblioteca",
      "repositorio",
      "ebsco",
      "scopus",
      "digital",
      "presencial"
    ]
  },
  {
    "id": "SER-002",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo obtengo mi correo institucional de la UNSAAC?",
    "respuesta": "El correo institucional se activa en la Dirección de Tecnologías de la Información (DTI) presentando el carné universitario. El formato es nombre.apellido@unsaac.edu.pe. Con este correo se accede a plataformas como Google Workspace for Education, Microsoft 365 y las aulas virtuales.",
    "intent": "consulta_ser_correo_institucional",
    "variaciones": [
      "¿Cómo activo mi email universitario?",
      "¿Cómo saco mi correo de la UNSAAC?",
      "¿Qué correo me da la universidad?"
    ],
    "fuente": "Dirección de Tecnologías de la Información (DTI) UNSAAC",
    "tags": [
      "correo",
      "email",
      "institucional",
      "dti",
      "google workspace",
      "microsoft"
    ]
  },
  {
    "id": "SER-003",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Dónde solicito mis certificados y constancias académicas?",
    "respuesta": "Los certificados y constancias se tramitan en la DRAA. Algunos documentos como la constancia de matrícula pueden descargarse directamente desde el portal SERUNSA. El tiempo de entrega varía de 3 a 10 días hábiles según el tipo de documento.",
    "intent": "consulta_ser_certificados_constancias",
    "variaciones": [
      "¿Cómo pido una constancia de estudios?",
      "¿Dónde tramito mis certificados de notas?",
      "¿Dónde saco el certificado de matrícula?"
    ],
    "fuente": "Dirección de Registros Académicos y Admisión (DRAA)",
    "tags": [
      "certificados",
      "constancias",
      "draa",
      "serunsa",
      "dias habiles"
    ]
  },
  {
    "id": "SER-004",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo puedo acceder al aula virtual de la UNSAAC?",
    "respuesta": "El aula virtual está disponible en el portal SERUNSA o en la plataforma Moodle institucional (aulavirtual.unsaac.edu.pe). Se ingresa con usuario y contraseña institucional. Ahí se encuentran materiales de cursos, tareas, foros y comunicados de los docentes.",
    "intent": "consulta_ser_aula_virtual",
    "variaciones": [
      "¿Dónde está el aula virtual de la universidad?",
      "¿Cómo ingreso al campus virtual?",
      "¿Cuál es la plataforma de e-learning de la UNSAAC?"
    ],
    "fuente": "Dirección de Tecnologías de la Información (DTI) UNSAAC",
    "tags": [
      "aula virtual",
      "moodle",
      "serunsa",
      "campus virtual",
      "elearning",
      "plataforma"
    ]
  },
  {
    "id": "SER-005",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿La UNSAAC cuenta con laboratorios de cómputo disponibles para estudiantes?",
    "respuesta": "La EPIIS cuenta con cuatro laboratorios de cómputo equipados con estaciones de trabajo actualizadas, software especializado (Visual Studio, Eclipse, NetBeans, MATLAB, Cisco Packet Tracer, entre otros) y acceso a internet. El uso está disponible para estudiantes matriculados durante el horario de atención publicado.",
    "intent": "consulta_ser_laboratorios_computo",
    "variaciones": [
      "¿Hay computadoras para usar en la universidad?",
      "¿La escuela de sistemas tiene laboratorios?",
      "¿Puedo usar los laboratorios de la EPIIS?"
    ],
    "fuente": "EPIIS - Dirección de Escuela Profesional",
    "tags": [
      "laboratorios",
      "computo",
      "visual studio",
      "matlab",
      "cisco",
      "cuatro laboratorios"
    ]
  },
  {
    "id": "SER-006",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo solicito una carta de presentación para eventos o concursos?",
    "respuesta": "Para obtener una carta de presentación debes presentar una solicitud en la secretaría de la escuela, especificando el evento, la institución organizadora y la fecha. La carta es emitida por el Director de la Escuela en un plazo de tres días hábiles, previo visto bueno de la Dirección de la Facultad.",
    "intent": "consulta_ser_carta_presentacion",
    "variaciones": [
      "¿Cómo pido una carta de la universidad para un concurso?",
      "¿La escuela emite cartas de presentación?",
      "¿Qué hago para que la universidad me represente en un evento?"
    ],
    "fuente": "Dirección de la Escuela Profesional EPIIS",
    "tags": [
      "carta",
      "presentacion",
      "concurso",
      "evento",
      "tres dias",
      "secretaria"
    ]
  },
  {
    "id": "SER-007",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Existe un servicio de impresión o fotocopiado dentro del campus?",
    "respuesta": "El campus cuenta con centros de fotocopiado e impresión en distintos puntos, incluyendo los pabellones de la Facultad de Ingeniería y los alrededores de la Biblioteca Central. Algunos laboratorios de la escuela permiten impresión con el carné universitario, sujeto a disponibilidad y cuotas establecidas.",
    "intent": "consulta_ser_impresion_campus",
    "variaciones": [
      "¿Hay fotocopiadoras en la universidad?",
      "¿Puedo imprimir en el campus?",
      "¿Dónde imprimo mis documentos en la UNSAAC?"
    ],
    "fuente": "UNSAAC - Campus Universitario",
    "tags": [
      "impresion",
      "fotocopiado",
      "campus",
      "carne",
      "laboratorios",
      "cuota"
    ]
  },
  {
    "id": "SER-008",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo puedo obtener mi récord de notas oficial?",
    "respuesta": "El récord de notas oficial se tramita en la DRAA presentando la solicitud correspondiente y pagando la tasa establecida en el TUPA de la universidad. En el portal SERUNSA puedes visualizar e imprimir un reporte de notas no oficial. El documento oficial lleva firma y sello de la Dirección de Registros.",
    "intent": "consulta_ser_record_notas",
    "variaciones": [
      "¿Dónde saco el récord de notas?",
      "¿Cómo pido mis notas certificadas?",
      "¿Qué es el récord de notas y cómo lo tramito?"
    ],
    "fuente": "Dirección de Registros Académicos y Admisión (DRAA)",
    "tags": [
      "record",
      "notas",
      "oficial",
      "draa",
      "tupa",
      "serunsa"
    ]
  },
  {
    "id": "SER-009",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Qué es el TUPA y dónde puedo consultarlo?",
    "respuesta": "El TUPA (Texto Único de Procedimientos Administrativos) es el documento oficial que detalla todos los trámites y servicios ofrecidos por la UNSAAC, incluyendo requisitos, plazos y costos. Está disponible en la página web de la universidad en la sección 'Transparencia' y en la secretaría general.",
    "intent": "consulta_ser_tupa",
    "variaciones": [
      "¿Qué significa TUPA?",
      "¿Dónde veo los costos de los trámites?",
      "¿Cómo sé cuánto cuesta cada trámite?"
    ],
    "fuente": "Secretaría General UNSAAC",
    "tags": [
      "tupa",
      "tramites",
      "costos",
      "requisitos",
      "transparencia",
      "procedimientos"
    ]
  },
  {
    "id": "SER-010",
    "categoria": "SER",
    "categoria_nombre": "Servicios Universitarios",
    "pregunta": "¿Cómo me comunico con la secretaría de la Escuela de Ingeniería Informática?",
    "respuesta": "La secretaría de la EPIIS está ubicada en el pabellón de Ingeniería del Campus Universitario de Perayoc. El horario de atención es de lunes a viernes de 8:00 a 13:00 y de 15:00 a 18:00 horas. Puedes comunicarte por correo institucional o llamar a la central telefónica de la Facultad de Ingeniería.",
    "intent": "consulta_ser_contacto_secretaria",
    "variaciones": [
      "¿Cuál es el correo de la secretaría de sistemas?",
      "¿Cómo contacto a la secretaría de la EPIIS?",
      "¿Dónde está la oficina de la escuela de informática?"
    ],
    "fuente": "Secretaría EPIIS",
    "tags": [
      "secretaria",
      "epiis",
      "perayoc",
      "horario",
      "correo",
      "contacto"
    ]
  }
]

INTENTS_DATA = {
    "version": "1.1.0",
    "proyecto": "Chatbot Académico EPIIS-UNSAAC",
    "total_intents": 100,
    "intents": [
  {
    "intent_id": "consulta_tutoria_definicion",
    "display_name": "Qué es la tutoría académica en la UNSAAC",
    "categoria": "TUT",
    "corpus_ref": "TUT-001",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_frecuencia",
    "display_name": "Con qué frecuencia debo asistir a las sesiones de tutoría",
    "categoria": "TUT",
    "corpus_ref": "TUT-002",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_contacto_tutor",
    "display_name": "Quién es mi tutor asignado y cómo puedo contactarlo",
    "categoria": "TUT",
    "corpus_ref": "TUT-003",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_temas",
    "display_name": "Qué temas se tratan en las sesiones de tutoría",
    "categoria": "TUT",
    "corpus_ref": "TUT-004",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_obligatoriedad",
    "display_name": "La asistencia a tutoría es obligatoria",
    "categoria": "TUT",
    "corpus_ref": "TUT-005",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_registro_asistencia",
    "display_name": "Cómo se registra la asistencia a las sesiones de tutoría",
    "categoria": "TUT",
    "corpus_ref": "TUT-006",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_cambio_tutor",
    "display_name": "Puedo cambiar de tutor si tengo dificultades con el asignado",
    "categoria": "TUT",
    "corpus_ref": "TUT-007",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_reglamento",
    "display_name": "Existe algún reglamento que regule el proceso de tutorías",
    "categoria": "TUT",
    "corpus_ref": "TUT-008",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_materiales_sesion",
    "display_name": "Qué debo llevar a una sesión de tutoría",
    "categoria": "TUT",
    "corpus_ref": "TUT-009",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tutoria_confidencialidad",
    "display_name": "Las sesiones de tutoría son confidenciales",
    "categoria": "TUT",
    "corpus_ref": "TUT-010",
    "confianza_minima": 0.72,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_general",
    "display_name": "Cuáles son los tipos de tutoría que ofrece la escuela",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-001",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_individual",
    "display_name": "En qué consiste la tutoría individual",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-002",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_grupal",
    "display_name": "Qué es la tutoría grupal y cómo se organiza",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-003",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_pares",
    "display_name": "Qué es la tutoría entre pares",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-004",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_virtual",
    "display_name": "Existe tutoría en línea o virtual",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-005",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_vs_psicologia",
    "display_name": "Cuál es la diferencia entre tutoría académica y orientación ",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-006",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_ingresantes",
    "display_name": "Qué es la tutoría de ingresantes",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-007",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_riesgo_academico",
    "display_name": "Existe tutoría para estudiantes con bajo rendimiento",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-008",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_vs_asesoria_tesis",
    "display_name": "Qué diferencia hay entre tutoría y asesoría de tesis",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-009",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_tipos_tutoria_vocacional",
    "display_name": "La tutoría cubre también orientación vocacional y de carrera",
    "categoria": "TTUT",
    "corpus_ref": "TTUT-010",
    "confianza_minima": 0.7,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_duracion_carrera",
    "display_name": "Cuántos semestres tiene la carrera de Ingeniería Informática",
    "categoria": "CUR",
    "corpus_ref": "CUR-001",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_total_creditos",
    "display_name": "Cuántos créditos son necesarios para culminar la carrera",
    "categoria": "CUR",
    "corpus_ref": "CUR-002",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_primer_semestre",
    "display_name": "Cuáles son los cursos del primer semestre",
    "categoria": "CUR",
    "corpus_ref": "CUR-003",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_adelanto_semestres",
    "display_name": "Puedo llevar cursos de otro semestre si aprobé los prerrequi",
    "categoria": "CUR",
    "corpus_ref": "CUR-004",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_desaprobacion",
    "display_name": "Qué pasa si desapruebo un curso",
    "categoria": "CUR",
    "corpus_ref": "CUR-005",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_areas_plan_estudios",
    "display_name": "Cómo está organizado el plan de estudios por áreas",
    "categoria": "CUR",
    "corpus_ref": "CUR-006",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_electivos",
    "display_name": "Cuántos cursos electivos debo llevar durante la carrera",
    "categoria": "CUR",
    "corpus_ref": "CUR-007",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_horario_clases",
    "display_name": "Dónde puedo ver el horario de clases del semestre",
    "categoria": "CUR",
    "corpus_ref": "CUR-008",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_creditos_definicion",
    "display_name": "Qué son los créditos académicos y cómo se calculan",
    "categoria": "CUR",
    "corpus_ref": "CUR-009",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_cursos_promedio_ponderado",
    "display_name": "Cómo se calcula el promedio ponderado semestral",
    "categoria": "CUR",
    "corpus_ref": "CUR-010",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_listado_especialidades",
    "display_name": "Cuáles son las especialidades que ofrece la Escuela de Ingen",
    "categoria": "ESP",
    "corpus_ref": "ESP-001",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_cuando_elegir",
    "display_name": "Cuándo debo elegir mi especialidad",
    "categoria": "ESP",
    "corpus_ref": "ESP-002",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_demanda_laboral",
    "display_name": "Qué especialidad tiene más demanda laboral en la región",
    "categoria": "ESP",
    "corpus_ref": "ESP-003",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_cursos_ingenieria_software",
    "display_name": "Cuáles son los cursos de la especialidad de Ingeniería de So",
    "categoria": "ESP",
    "corpus_ref": "ESP-004",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_cambio_especialidad",
    "display_name": "Puedo cambiar de especialidad una vez que la elegí",
    "categoria": "ESP",
    "corpus_ref": "ESP-005",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_cursos_redes_telecom",
    "display_name": "Qué cursos incluye la especialidad de Redes y Telecomunicaci",
    "categoria": "ESP",
    "corpus_ref": "ESP-006",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_ia_ciencia_datos",
    "display_name": "Qué es la especialidad de Inteligencia Artificial y Ciencia ",
    "categoria": "ESP",
    "corpus_ref": "ESP-007",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_seguridad_certificaciones",
    "display_name": "La especialidad de Seguridad Informática tiene certificacion",
    "categoria": "ESP",
    "corpus_ref": "ESP-008",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_convenios_practicas",
    "display_name": "Las especialidades tienen convenios con empresas para prácti",
    "categoria": "ESP",
    "corpus_ref": "ESP-009",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_esp_electivos_cruzados",
    "display_name": "Puedo llevar cursos de otras especialidades como electivos",
    "categoria": "ESP",
    "corpus_ref": "ESP-010",
    "confianza_minima": 0.73,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ppp_requisitos_inicio",
    "display_name": "Cuáles son los requisitos para iniciar las prácticas pre-pro",
    "categoria": "PPP",
    "corpus_ref": "PPP-001",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_horas_acumuladas",
    "display_name": "Cuántas horas de prácticas debo acumular",
    "categoria": "PPP",
    "corpus_ref": "PPP-002",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_empresa_convenio",
    "display_name": "La empresa donde hago prácticas debe tener convenio con la U",
    "categoria": "PPP",
    "corpus_ref": "PPP-003",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_carta_presentacion",
    "display_name": "Cómo consigo la carta de presentación para las prácticas",
    "categoria": "PPP",
    "corpus_ref": "PPP-004",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_informe_final",
    "display_name": "Qué es el informe de prácticas y cuándo debo presentarlo",
    "categoria": "PPP",
    "corpus_ref": "PPP-005",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_remuneracion",
    "display_name": "Las prácticas pre-profesionales son remuneradas",
    "categoria": "PPP",
    "corpus_ref": "PPP-006",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_sector_publico",
    "display_name": "Puedo hacer mis prácticas en una empresa pública o instituci",
    "categoria": "PPP",
    "corpus_ref": "PPP-007",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_supervision_universidad",
    "display_name": "Quién supervisa y evalúa mis prácticas desde la universidad",
    "categoria": "PPP",
    "corpus_ref": "PPP-008",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_trabajo_actual",
    "display_name": "Puedo hacer las prácticas en la misma empresa donde trabajo",
    "categoria": "PPP",
    "corpus_ref": "PPP-009",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ppp_requisito_titulacion",
    "display_name": "Las prácticas pre-profesionales son un requisito para titula",
    "categoria": "PPP",
    "corpus_ref": "PPP-010",
    "confianza_minima": 0.76,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_bie_servicios_generales",
    "display_name": "Qué servicios de bienestar ofrece la UNSAAC a sus estudiante",
    "categoria": "BIE",
    "corpus_ref": "BIE-001",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_atencion_psicologica",
    "display_name": "Cómo puedo acceder a la atención psicológica gratuita",
    "categoria": "BIE",
    "corpus_ref": "BIE-002",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_beca_alimentacion",
    "display_name": "Qué es la beca de alimentación y cómo la solicito",
    "categoria": "BIE",
    "corpus_ref": "BIE-003",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_apoyo_economico",
    "display_name": "Existe apoyo económico para estudiantes en situación de vuln",
    "categoria": "BIE",
    "corpus_ref": "BIE-004",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_seguro_estudiantil",
    "display_name": "Cómo funciona el seguro estudiantil de la UNSAAC",
    "categoria": "BIE",
    "corpus_ref": "BIE-005",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_deportes_cultura",
    "display_name": "Qué deportes y actividades culturales puedo practicar en la ",
    "categoria": "BIE",
    "corpus_ref": "BIE-006",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_residencia_universitaria",
    "display_name": "La UNSAAC tiene residencia universitaria para estudiantes de",
    "categoria": "BIE",
    "corpus_ref": "BIE-007",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_certificado_socioeconomico",
    "display_name": "Cómo solicito un certificado de situación socioeconómica",
    "categoria": "BIE",
    "corpus_ref": "BIE-008",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_comedor_acceso",
    "display_name": "El comedor universitario está disponible para todos los estu",
    "categoria": "BIE",
    "corpus_ref": "BIE-009",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_bie_contacto_oficina",
    "display_name": "A qué número o correo puedo comunicarme con Bienestar Univer",
    "categoria": "BIE",
    "corpus_ref": "BIE-010",
    "confianza_minima": 0.74,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_mov_programas_disponibles",
    "display_name": "Qué programas de movilidad estudiantil ofrece la UNSAAC",
    "categoria": "MOV",
    "corpus_ref": "MOV-001",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_requisitos_intercambio",
    "display_name": "Cuáles son los requisitos mínimos para participar en un prog",
    "categoria": "MOV",
    "corpus_ref": "MOV-002",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_convalidacion_cursos",
    "display_name": "Los cursos llevados en el intercambio se convalidan en la UN",
    "categoria": "MOV",
    "corpus_ref": "MOV-003",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_apoyo_economico",
    "display_name": "Hay apoyo económico para los estudiantes que van de intercam",
    "categoria": "MOV",
    "corpus_ref": "MOV-004",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_estudiantes_entrantes",
    "display_name": "Puedo recibir estudiantes de intercambio de otras universida",
    "categoria": "MOV",
    "corpus_ref": "MOV-005",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_convocatorias",
    "display_name": "Dónde puedo enterarme de las convocatorias de movilidad estu",
    "categoria": "MOV",
    "corpus_ref": "MOV-006",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_documentos_postulacion",
    "display_name": "Qué documentos necesito para postular a la movilidad estudia",
    "categoria": "MOV",
    "corpus_ref": "MOV-007",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_impacto_titulacion",
    "display_name": "El intercambio afecta mi proceso de titulación o retrasa mi ",
    "categoria": "MOV",
    "corpus_ref": "MOV-008",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_requisito_idioma",
    "display_name": "Necesito dominar otro idioma para participar en la movilidad",
    "categoria": "MOV",
    "corpus_ref": "MOV-009",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mov_universidades_convenio",
    "display_name": "La UNSAAC tiene convenios con universidades reconocidas inte",
    "categoria": "MOV",
    "corpus_ref": "MOV-010",
    "confianza_minima": 0.68,
    "prioridad": "media"
  },
  {
    "intent_id": "consulta_mat_fechas_matricula",
    "display_name": "Cuándo se realizan las matrículas en la UNSAAC",
    "categoria": "MAT",
    "corpus_ref": "MAT-001",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_proceso_serunsa",
    "display_name": "Cómo me matriculo en el portal SERUNSA",
    "categoria": "MAT",
    "corpus_ref": "MAT-002",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_requisitos",
    "display_name": "Cuáles son los requisitos para poder matricularme",
    "categoria": "MAT",
    "corpus_ref": "MAT-003",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_modalidad_presencial",
    "display_name": "Puedo hacer la matrícula de forma presencial",
    "categoria": "MAT",
    "corpus_ref": "MAT-004",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_fuera_de_plazo",
    "display_name": "Qué hago si no pude matricularme dentro del plazo establecid",
    "categoria": "MAT",
    "corpus_ref": "MAT-005",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_creditos_maximo",
    "display_name": "Cuántos créditos máximos puedo llevar por semestre",
    "categoria": "MAT",
    "corpus_ref": "MAT-006",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_retiro_curso",
    "display_name": "Puedo retirar un curso después de matricularme",
    "categoria": "MAT",
    "corpus_ref": "MAT-007",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_condicional",
    "display_name": "Qué es la matrícula condicional y cuándo aplica",
    "categoria": "MAT",
    "corpus_ref": "MAT-008",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_costo",
    "display_name": "Cuánto cuesta la matrícula en la UNSAAC",
    "categoria": "MAT",
    "corpus_ref": "MAT-009",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_mat_seguro_activacion",
    "display_name": "Qué es el seguro universitario y cuándo se activa con la mat",
    "categoria": "MAT",
    "corpus_ref": "MAT-010",
    "confianza_minima": 0.8,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_modalidades",
    "display_name": "Cuáles son las modalidades de titulación disponibles en la E",
    "categoria": "TIT",
    "corpus_ref": "TIT-001",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_requisitos_inicio",
    "display_name": "Cuáles son los requisitos para iniciar el proceso de titulac",
    "categoria": "TIT",
    "corpus_ref": "TIT-002",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_grado_bachiller",
    "display_name": "Cómo obtengo el grado de Bachiller",
    "categoria": "TIT",
    "corpus_ref": "TIT-003",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_plazo_titulacion",
    "display_name": "Cuánto tiempo tengo para titularme después de terminar los c",
    "categoria": "TIT",
    "corpus_ref": "TIT-004",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_suficiencia_profesional",
    "display_name": "Qué es un trabajo de suficiencia profesional",
    "categoria": "TIT",
    "corpus_ref": "TIT-005",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_eleccion_asesor",
    "display_name": "Cómo escojo a mi asesor de tesis",
    "categoria": "TIT",
    "corpus_ref": "TIT-006",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_duracion_proceso",
    "display_name": "Cuánto tiempo toma habitualmente el proceso de titulación",
    "categoria": "TIT",
    "corpus_ref": "TIT-007",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_jurado_sustentacion",
    "display_name": "En qué consiste el jurado de sustentación",
    "categoria": "TIT",
    "corpus_ref": "TIT-008",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_desaprobacion_sustentacion",
    "display_name": "Qué pasa si desapruebo la sustentación",
    "categoria": "TIT",
    "corpus_ref": "TIT-009",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_tit_registro_sunedu",
    "display_name": "Dónde debo registrar mi título una vez obtenido",
    "categoria": "TIT",
    "corpus_ref": "TIT-010",
    "confianza_minima": 0.78,
    "prioridad": "muy_alta"
  },
  {
    "intent_id": "consulta_ser_biblioteca",
    "display_name": "Cómo accedo a los recursos de la biblioteca universitaria",
    "categoria": "SER",
    "corpus_ref": "SER-001",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_correo_institucional",
    "display_name": "Cómo obtengo mi correo institucional de la UNSAAC",
    "categoria": "SER",
    "corpus_ref": "SER-002",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_certificados_constancias",
    "display_name": "Dónde solicito mis certificados y constancias académicas",
    "categoria": "SER",
    "corpus_ref": "SER-003",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_aula_virtual",
    "display_name": "Cómo puedo acceder al aula virtual de la UNSAAC",
    "categoria": "SER",
    "corpus_ref": "SER-004",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_laboratorios_computo",
    "display_name": "La UNSAAC cuenta con laboratorios de cómputo disponibles par",
    "categoria": "SER",
    "corpus_ref": "SER-005",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_carta_presentacion",
    "display_name": "Cómo solicito una carta de presentación para eventos o concu",
    "categoria": "SER",
    "corpus_ref": "SER-006",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_impresion_campus",
    "display_name": "Existe un servicio de impresión o fotocopiado dentro del cam",
    "categoria": "SER",
    "corpus_ref": "SER-007",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_record_notas",
    "display_name": "Cómo puedo obtener mi récord de notas oficial",
    "categoria": "SER",
    "corpus_ref": "SER-008",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_tupa",
    "display_name": "Qué es el TUPA y dónde puedo consultarlo",
    "categoria": "SER",
    "corpus_ref": "SER-009",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  },
  {
    "intent_id": "consulta_ser_contacto_secretaria",
    "display_name": "Cómo me comunico con la secretaría de la Escuela de Ingenier",
    "categoria": "SER",
    "corpus_ref": "SER-010",
    "confianza_minima": 0.76,
    "prioridad": "alta"
  }
]
}

KEYWORDS_DATA = {
  "version": "1.1.0",
  "descripcion": "Keywords completos para 100 intents — versión mejorada para scoring ponderado",
  "keywords": [
    {
      "intent_id": "consulta_tutoria_definicion",
      "categoria": "TUT",
      "keywords_primarios": [
        "tutoria",
        "tutoria academica",
        "acompanamiento"
      ],
      "keywords_secundarios": [
        "docente tutor",
        "orientacion",
        "seguimiento"
      ],
      "sinonimos": [
        "tutoreo",
        "mentoria"
      ],
      "frases_trigger": [
        "que es la tutoria",
        "en que consiste tutoria",
        "como funciona tutoria"
      ]
    },
    {
      "intent_id": "consulta_tutoria_frecuencia",
      "categoria": "TUT",
      "keywords_primarios": [
        "frecuencia",
        "veces",
        "sesiones",
        "cada cuanto"
      ],
      "keywords_secundarios": [
        "reuniones",
        "citas",
        "semestre",
        "encuentros"
      ],
      "sinonimos": [
        "periodicidad"
      ],
      "frases_trigger": [
        "cuantas veces tutoria",
        "frecuencia sesiones",
        "cada cuanto tiempo tutor"
      ]
    },
    {
      "intent_id": "consulta_tutoria_contacto_tutor",
      "categoria": "TUT",
      "keywords_primarios": [
        "tutor asignado",
        "contactar tutor",
        "quien tutor"
      ],
      "keywords_secundarios": [
        "serunsa",
        "correo",
        "horario atencion"
      ],
      "sinonimos": [
        "tutor academico"
      ],
      "frases_trigger": [
        "quien es tutor",
        "como contacto tutor",
        "donde encuentro tutor"
      ]
    },
    {
      "intent_id": "consulta_tutoria_temas",
      "categoria": "TUT",
      "keywords_primarios": [
        "temas tutoria",
        "que se trata",
        "contenidos"
      ],
      "keywords_secundarios": [
        "academica",
        "personal",
        "profesional",
        "dimensiones"
      ],
      "sinonimos": [
        "materias tutoria"
      ],
      "frases_trigger": [
        "que temas tutoria",
        "sobre que tutoria",
        "contenidos tutoría"
      ]
    },
    {
      "intent_id": "consulta_tutoria_obligatoriedad",
      "categoria": "TUT",
      "keywords_primarios": [
        "obligatoria",
        "obligatorio",
        "asistencia"
      ],
      "keywords_secundarios": [
        "sancion",
        "comite tutorial",
        "condicionada"
      ],
      "sinonimos": [
        "forzoso"
      ],
      "frases_trigger": [
        "es obligatoria tutoria",
        "puedo faltar tutoria",
        "sancion tutoria"
      ]
    },
    {
      "intent_id": "consulta_tutoria_registro_asistencia",
      "categoria": "TUT",
      "keywords_primarios": [
        "registro",
        "asistencia",
        "lista"
      ],
      "keywords_secundarios": [
        "plataforma",
        "control",
        "valoracion"
      ],
      "sinonimos": [
        "control asistencia"
      ],
      "frases_trigger": [
        "como se registra tutoria",
        "hay lista tutoria",
        "control asistencia tutoria"
      ]
    },
    {
      "intent_id": "consulta_tutoria_cambio_tutor",
      "categoria": "TUT",
      "keywords_primarios": [
        "cambiar tutor",
        "cambio de tutor",
        "otro tutor"
      ],
      "keywords_secundarios": [
        "solicitud",
        "comite",
        "vicerrectorado"
      ],
      "sinonimos": [
        "solicitar tutor"
      ],
      "frases_trigger": [
        "puedo cambiar tutor",
        "como cambio tutor",
        "solicitar cambio tutor"
      ]
    },
    {
      "intent_id": "consulta_tutoria_reglamento",
      "categoria": "TUT",
      "keywords_primarios": [
        "reglamento",
        "norma",
        "resolucion",
        "ley"
      ],
      "keywords_secundarios": [
        "cu-0220",
        "2017",
        "ley universitaria"
      ],
      "sinonimos": [
        "normativa tutoria"
      ],
      "frases_trigger": [
        "reglamento tutorias",
        "normas tutoria",
        "documento tutoria"
      ]
    },
    {
      "intent_id": "consulta_tutoria_materiales_sesion",
      "categoria": "TUT",
      "keywords_primarios": [
        "llevar",
        "documentos",
        "carne"
      ],
      "keywords_secundarios": [
        "notas",
        "horario",
        "preparar"
      ],
      "sinonimos": [
        "que llevar tutoria"
      ],
      "frases_trigger": [
        "que debo llevar tutoria",
        "que necesito tutoria",
        "documentos tutoria"
      ]
    },
    {
      "intent_id": "consulta_tutoria_confidencialidad",
      "categoria": "TUT",
      "keywords_primarios": [
        "confidencial",
        "privado",
        "secreto"
      ],
      "keywords_secundarios": [
        "datos",
        "informacion",
        "padres"
      ],
      "sinonimos": [
        "privacidad"
      ],
      "frases_trigger": [
        "tutoria confidencial",
        "privada informacion tutoria",
        "comparte informacion tutor"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_general",
      "categoria": "TUT",
      "keywords_primarios": [
        "tipos tutoria",
        "modalidades tutoria",
        "clases tutoria"
      ],
      "keywords_secundarios": [
        "individual",
        "grupal",
        "pares"
      ],
      "sinonimos": [
        "formas tutoria"
      ],
      "frases_trigger": [
        "que tipos tutoria",
        "modalidades tutoria",
        "cuantas clases tutoria"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_individual",
      "categoria": "TUT",
      "keywords_primarios": [
        "individual",
        "personalizada",
        "uno a uno"
      ],
      "keywords_secundarios": [
        "privada",
        "dificultades",
        "seguimiento"
      ],
      "sinonimos": [
        "sesion privada"
      ],
      "frases_trigger": [
        "tutoria individual",
        "tutoria personalizada",
        "tutoria uno a uno"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_grupal",
      "categoria": "TUT",
      "keywords_primarios": [
        "grupal",
        "grupo",
        "25 estudiantes"
      ],
      "keywords_secundarios": [
        "semestre",
        "docente",
        "rendimiento"
      ],
      "sinonimos": [
        "tutoría grupal"
      ],
      "frases_trigger": [
        "tutoria grupal",
        "como se organiza grupal",
        "cuantos en tutoria grupal"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_pares",
      "categoria": "TUT",
      "keywords_primarios": [
        "pares",
        "compañeros",
        "entre estudiantes"
      ],
      "keywords_secundarios": [
        "avanzados",
        "voluntaria",
        "supervision"
      ],
      "sinonimos": [
        "tutoreo pares"
      ],
      "frases_trigger": [
        "tutoria entre pares",
        "tutores pares",
        "compañero tutor"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_virtual",
      "categoria": "TUT",
      "keywords_primarios": [
        "virtual",
        "online",
        "videoconferencia",
        "zoom"
      ],
      "keywords_secundarios": [
        "distancia",
        "google meet",
        "presencial"
      ],
      "sinonimos": [
        "tutoria remota"
      ],
      "frases_trigger": [
        "tutoria virtual",
        "tutoria online",
        "tutoria por videoconferencia"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_vs_psicologia",
      "categoria": "TUT",
      "keywords_primarios": [
        "psicologo",
        "psicologia",
        "salud mental"
      ],
      "keywords_secundarios": [
        "diferencia",
        "bienestar",
        "derivar"
      ],
      "sinonimos": [
        "orientacion psicologica"
      ],
      "frases_trigger": [
        "diferencia tutoria psicologo",
        "tutoria incluye psicologia",
        "cuando ir psicologo"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_ingresantes",
      "categoria": "TUT",
      "keywords_primarios": [
        "ingresantes",
        "nuevos",
        "primer semestre",
        "adaptacion"
      ],
      "keywords_secundarios": [
        "transicion",
        "abandono",
        "reforzado"
      ],
      "sinonimos": [
        "tutoria primer año"
      ],
      "frases_trigger": [
        "tutoria ingresantes",
        "tutoria primer semestre",
        "nuevos alumnos tutoria"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_riesgo_academico",
      "categoria": "TUT",
      "keywords_primarios": [
        "bajo rendimiento",
        "riesgo academico",
        "notas bajas"
      ],
      "keywords_secundarios": [
        "condicionada",
        "reforzada",
        "jalado"
      ],
      "sinonimos": [
        "tutoria riesgo"
      ],
      "frases_trigger": [
        "tutoria bajo rendimiento",
        "jale cursos tutoria",
        "riesgo academico tutoria"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_vs_asesoria_tesis",
      "categoria": "TUT",
      "keywords_primarios": [
        "tesis",
        "asesoria",
        "diferencia tesis tutoria"
      ],
      "keywords_secundarios": [
        "asesor",
        "investigacion",
        "director escuela"
      ],
      "sinonimos": [
        "asesor tesis vs tutor"
      ],
      "frases_trigger": [
        "diferencia tutoria tesis",
        "tutor ayuda tesis",
        "asesor tesis"
      ]
    },
    {
      "intent_id": "consulta_tipos_tutoria_vocacional",
      "categoria": "TUT",
      "keywords_primarios": [
        "vocacional",
        "carrera profesional",
        "futuro"
      ],
      "keywords_secundarios": [
        "laboral",
        "orientacion",
        "preparacion"
      ],
      "sinonimos": [
        "orientacion vocacional"
      ],
      "frases_trigger": [
        "tutoria vocacional",
        "orientacion carrera tutoria",
        "futuro profesional tutoria"
      ]
    },
    {
      "intent_id": "consulta_cursos_duracion_carrera",
      "categoria": "CUR",
      "keywords_primarios": [
        "semestres",
        "duracion",
        "anos",
        "ciclos"
      ],
      "keywords_secundarios": [
        "10 semestres",
        "5 anos",
        "carrera"
      ],
      "sinonimos": [
        "duracion estudios"
      ],
      "frases_trigger": [
        "cuantos semestres carrera",
        "cuantos anos carrera",
        "duracion carrera sistemas"
      ]
    },
    {
      "intent_id": "consulta_cursos_total_creditos",
      "categoria": "CUR",
      "keywords_primarios": [
        "creditos",
        "total creditos",
        "220"
      ],
      "keywords_secundarios": [
        "plan estudios",
        "curriculo",
        "culminar"
      ],
      "sinonimos": [
        "cuantos creditos"
      ],
      "frases_trigger": [
        "cuantos creditos carrera",
        "total creditos carrera",
        "creditos titulacion"
      ]
    },
    {
      "intent_id": "consulta_cursos_primer_semestre",
      "categoria": "CUR",
      "keywords_primarios": [
        "primer semestre",
        "primer ciclo",
        "cursos inicio"
      ],
      "keywords_secundarios": [
        "matematica",
        "programacion",
        "algorítmica"
      ],
      "sinonimos": [
        "materias inicio"
      ],
      "frases_trigger": [
        "cursos primer semestre",
        "que llevo primer ciclo",
        "materias inicio carrera"
      ]
    },
    {
      "intent_id": "consulta_cursos_adelanto_semestres",
      "categoria": "CUR",
      "keywords_primarios": [
        "adelantar",
        "prerrequisitos",
        "avanzados"
      ],
      "keywords_secundarios": [
        "24 creditos",
        "autorizacion",
        "ciclos superiores"
      ],
      "sinonimos": [
        "adelanto cursos"
      ],
      "frases_trigger": [
        "puedo adelantar cursos",
        "llevar cursos avanzados",
        "prerrequisitos cursos"
      ]
    },
    {
      "intent_id": "consulta_cursos_desaprobacion",
      "categoria": "CUR",
      "keywords_primarios": [
        "desaprobacion",
        "jalar",
        "reprobar",
        "tres veces"
      ],
      "keywords_secundarios": [
        "consejo facultad",
        "solicitud",
        "ofertado"
      ],
      "sinonimos": [
        "reprobar curso"
      ],
      "frases_trigger": [
        "que pasa desapruebo curso",
        "consecuencias jalar",
        "cuantas veces puedo llevar curso"
      ]
    },
    {
      "intent_id": "consulta_cursos_areas_plan_estudios",
      "categoria": "CUR",
      "keywords_primarios": [
        "areas",
        "plan estudios",
        "organizacion"
      ],
      "keywords_secundarios": [
        "formacion general",
        "profesional",
        "electivos"
      ],
      "sinonimos": [
        "estructura curriculo"
      ],
      "frases_trigger": [
        "como organizado plan estudios",
        "areas curriculo",
        "grupos cursos"
      ]
    },
    {
      "intent_id": "consulta_cursos_electivos",
      "categoria": "CUR",
      "keywords_primarios": [
        "electivos",
        "optativas",
        "libre eleccion",
        "18 creditos"
      ],
      "keywords_secundarios": [
        "elegir",
        "ofertadas",
        "unidades academicas"
      ],
      "sinonimos": [
        "cursos libres"
      ],
      "frases_trigger": [
        "cuantos electivos",
        "materias optativas",
        "creditos electivos"
      ]
    },
    {
      "intent_id": "consulta_cursos_horario_clases",
      "categoria": "CUR",
      "keywords_primarios": [
        "horario",
        "clases",
        "aulas"
      ],
      "keywords_secundarios": [
        "serunsa",
        "publicado",
        "tablones"
      ],
      "sinonimos": [
        "ver horarios"
      ],
      "frases_trigger": [
        "donde ver horario",
        "como consulto horario",
        "horarios semestre"
      ]
    },
    {
      "intent_id": "consulta_cursos_creditos_definicion",
      "categoria": "CUR",
      "keywords_primarios": [
        "credito academico",
        "que es credito",
        "horas"
      ],
      "keywords_secundarios": [
        "silabo",
        "teórica",
        "laboratorio"
      ],
      "sinonimos": [
        "valor credito"
      ],
      "frases_trigger": [
        "que es credito academico",
        "como calcular credito",
        "un credito cuantas horas"
      ]
    },
    {
      "intent_id": "consulta_cursos_promedio_ponderado",
      "categoria": "CUR",
      "keywords_primarios": [
        "promedio",
        "ponderado",
        "calculo"
      ],
      "keywords_secundarios": [
        "vigesimal",
        "notas",
        "multiplicar"
      ],
      "sinonimos": [
        "promedio semestral"
      ],
      "frases_trigger": [
        "como calcular promedio",
        "promedio ponderado semestre",
        "calculo promedio"
      ]
    },
    {
      "intent_id": "consulta_esp_listado_especialidades",
      "categoria": "ESP",
      "keywords_primarios": [
        "especialidades",
        "especializaciones",
        "lineas"
      ],
      "keywords_secundarios": [
        "software",
        "redes",
        "ia",
        "seguridad"
      ],
      "sinonimos": [
        "especializacion carrera"
      ],
      "frases_trigger": [
        "cuales especialidades epiis",
        "que especializaciones hay",
        "areas especializacion"
      ]
    },
    {
      "intent_id": "consulta_esp_cuando_elegir",
      "categoria": "ESP",
      "keywords_primarios": [
        "cuando elegir",
        "vi semestre",
        "vii semestre"
      ],
      "keywords_secundarios": [
        "proceso seleccion",
        "promedio acumulado"
      ],
      "sinonimos": [
        "momento elegir especializacion"
      ],
      "frases_trigger": [
        "cuando escojo especialidad",
        "en que semestre elijo",
        "a partir de cuando me especializo"
      ]
    },
    {
      "intent_id": "consulta_esp_demanda_laboral",
      "categoria": "ESP",
      "keywords_primarios": [
        "demanda",
        "laboral",
        "trabajo",
        "mercado"
      ],
      "keywords_secundarios": [
        "empleabilidad",
        "cusco",
        "region"
      ],
      "sinonimos": [
        "mas solicitada"
      ],
      "frases_trigger": [
        "especialidad mas demanda",
        "que especialidad mas trabajo",
        "cual especialidad conviene"
      ]
    },
    {
      "intent_id": "consulta_esp_cursos_ingenieria_software",
      "categoria": "CUR",
      "keywords_primarios": [
        "ingenieria software",
        "software",
        "arquitectura"
      ],
      "keywords_secundarios": [
        "agil",
        "patrones",
        "pruebas",
        "movil"
      ],
      "sinonimos": [
        "cursos software"
      ],
      "frases_trigger": [
        "cursos ingenieria software",
        "que aprendo software",
        "materias software"
      ]
    },
    {
      "intent_id": "consulta_esp_cambio_especialidad",
      "categoria": "ESP",
      "keywords_primarios": [
        "cambiar especialidad",
        "cambio especializacion"
      ],
      "keywords_secundarios": [
        "solicitud",
        "vacantes",
        "una vez"
      ],
      "sinonimos": [
        "solicitar cambio"
      ],
      "frases_trigger": [
        "puedo cambiar especialidad",
        "cambio especializacion",
        "arrepentirme especialidad"
      ]
    },
    {
      "intent_id": "consulta_esp_cursos_redes_telecom",
      "categoria": "CUR",
      "keywords_primarios": [
        "redes",
        "telecomunicaciones",
        "networking"
      ],
      "keywords_secundarios": [
        "voip",
        "sdn",
        "inalambrico"
      ],
      "sinonimos": [
        "cursos redes"
      ],
      "frases_trigger": [
        "cursos redes telecomunicaciones",
        "que aprendo redes",
        "materias telecomunicaciones"
      ]
    },
    {
      "intent_id": "consulta_esp_ia_ciencia_datos",
      "categoria": "ESP",
      "keywords_primarios": [
        "inteligencia artificial",
        "ciencia datos",
        "machine learning"
      ],
      "keywords_secundarios": [
        "aprendizaje automatico",
        "mineria datos",
        "nlp"
      ],
      "sinonimos": [
        "especialidad ia"
      ],
      "frases_trigger": [
        "que es ia ciencia datos",
        "machine learning especialidad",
        "ia datos que estudia"
      ]
    },
    {
      "intent_id": "consulta_esp_seguridad_certificaciones",
      "categoria": "ESP",
      "keywords_primarios": [
        "seguridad",
        "ciberseguridad",
        "certificaciones"
      ],
      "keywords_secundarios": [
        "ceh",
        "cissp",
        "comptia"
      ],
      "sinonimos": [
        "certif seguridad"
      ],
      "frases_trigger": [
        "seguridad certificaciones",
        "ciberseguridad certificados",
        "certif etica hacker"
      ]
    },
    {
      "intent_id": "consulta_esp_convenios_practicas",
      "categoria": "ESP",
      "keywords_primarios": [
        "convenios empresas",
        "practicas empresa"
      ],
      "keywords_secundarios": [
        "sunat",
        "bcp",
        "mineras"
      ],
      "sinonimos": [
        "empresa practicas especialidad"
      ],
      "frases_trigger": [
        "convenios practicas especialidad",
        "empresas colaboradoras",
        "practicas por especialidad"
      ]
    },
    {
      "intent_id": "consulta_esp_electivos_cruzados",
      "categoria": "ESP",
      "keywords_primarios": [
        "electivos cruzados",
        "otra especialidad",
        "mezclar"
      ],
      "keywords_secundarios": [
        "interdisciplinario",
        "prerrequisitos",
        "cruce horarios"
      ],
      "sinonimos": [
        "cursos otra especialidad"
      ],
      "frases_trigger": [
        "electivos otra especialidad",
        "cursar otra linea",
        "mezclar especialidades"
      ]
    },
    {
      "intent_id": "consulta_ppp_requisitos_inicio",
      "categoria": "PPP",
      "keywords_primarios": [
        "requisitos practicas",
        "ppp",
        "ix semestre"
      ],
      "keywords_secundarios": [
        "promedio 11",
        "carta presentacion",
        "convenio"
      ],
      "sinonimos": [
        "condiciones practicas"
      ],
      "frases_trigger": [
        "requisitos practicas",
        "que necesito practicas",
        "condiciones ppp"
      ]
    },
    {
      "intent_id": "consulta_ppp_horas_acumuladas",
      "categoria": "PPP",
      "keywords_primarios": [
        "horas practicas",
        "300 horas",
        "duracion"
      ],
      "keywords_secundarios": [
        "tres meses",
        "tiempo completo"
      ],
      "sinonimos": [
        "cuantas horas ppp"
      ],
      "frases_trigger": [
        "cuantas horas practicas",
        "duracion practicas",
        "tiempo practicas"
      ]
    },
    {
      "intent_id": "consulta_ppp_empresa_convenio",
      "categoria": "PPP",
      "keywords_primarios": [
        "empresa convenio",
        "cualquier empresa"
      ],
      "keywords_secundarios": [
        "autorizacion",
        "constituida",
        "perfil carrera"
      ],
      "sinonimos": [
        "empresa practicas"
      ],
      "frases_trigger": [
        "empresa debe convenio",
        "donde hacer practicas",
        "empresa registrada"
      ]
    },
    {
      "intent_id": "consulta_ppp_carta_presentacion",
      "categoria": "PPP",
      "keywords_primarios": [
        "carta presentacion",
        "tramitar carta",
        "secretaria"
      ],
      "keywords_secundarios": [
        "tres dias",
        "constancia notas",
        "empresa receptora"
      ],
      "sinonimos": [
        "carta practicas"
      ],
      "frases_trigger": [
        "como consigo carta practicas",
        "donde tramito carta ppp",
        "quien emite carta"
      ]
    },
    {
      "intent_id": "consulta_ppp_informe_final",
      "categoria": "PPP",
      "keywords_primarios": [
        "informe practicas",
        "reporte",
        "informe final"
      ],
      "keywords_secundarios": [
        "formato",
        "unidad practicas",
        "finalizar"
      ],
      "sinonimos": [
        "informe ppp"
      ],
      "frases_trigger": [
        "que es informe practicas",
        "cuando entrego informe",
        "como es informe ppp"
      ]
    },
    {
      "intent_id": "consulta_ppp_remuneracion",
      "categoria": "PPP",
      "keywords_primarios": [
        "remuneracion",
        "sueldo",
        "pago",
        "rmv"
      ],
      "keywords_secundarios": [
        "50 por ciento",
        "ley 28518",
        "subvencion"
      ],
      "sinonimos": [
        "pagan practicas"
      ],
      "frases_trigger": [
        "me pagan practicas",
        "sueldo practicante",
        "practicas remuneradas"
      ]
    },
    {
      "intent_id": "consulta_ppp_sector_publico",
      "categoria": "PPP",
      "keywords_primarios": [
        "publico",
        "estado",
        "municipalidad",
        "ministerio"
      ],
      "keywords_secundarios": [
        "decreto legislativo",
        "gobierno regional"
      ],
      "sinonimos": [
        "entidades publicas"
      ],
      "frases_trigger": [
        "practicas sector publico",
        "practicar municipalidad",
        "ppp entidades estado"
      ]
    },
    {
      "intent_id": "consulta_ppp_supervision_universidad",
      "categoria": "PPP",
      "keywords_primarios": [
        "supervisa",
        "asesor practicas",
        "evalua"
      ],
      "keywords_secundarios": [
        "docente",
        "informe",
        "nota definitiva"
      ],
      "sinonimos": [
        "supervision ppp"
      ],
      "frases_trigger": [
        "quien supervisa practicas",
        "docente practicas",
        "calificacion practicas"
      ]
    },
    {
      "intent_id": "consulta_ppp_trabajo_actual",
      "categoria": "PPP",
      "keywords_primarios": [
        "donde trabajo",
        "empleo actual",
        "validar trabajo"
      ],
      "keywords_secundarios": [
        "autorizacion",
        "supervisor tecnico"
      ],
      "sinonimos": [
        "practicas empresa trabajo"
      ],
      "frases_trigger": [
        "practicas donde trabajo",
        "empleo como ppp",
        "validar trabajo practicas"
      ]
    },
    {
      "intent_id": "consulta_ppp_requisito_titulacion",
      "categoria": "PPP",
      "keywords_primarios": [
        "requisito titulacion",
        "obligatorio titularse"
      ],
      "keywords_secundarios": [
        "expediente",
        "certificado practicas"
      ],
      "sinonimos": [
        "ppp titulacion"
      ],
      "frases_trigger": [
        "practicas requisito titulo",
        "sin practicas titularme",
        "ppp obligatorio titulo"
      ]
    },
    {
      "intent_id": "consulta_bie_servicios_generales",
      "categoria": "BIE",
      "keywords_primarios": [
        "bienestar",
        "servicios",
        "apoyo"
      ],
      "keywords_secundarios": [
        "medico",
        "psicologo",
        "comedor",
        "seguro"
      ],
      "sinonimos": [
        "beneficios universitarios"
      ],
      "frases_trigger": [
        "que servicios bienestar",
        "ayudas universidad",
        "programas apoyo unsaac"
      ]
    },
    {
      "intent_id": "consulta_bie_atencion_psicologica",
      "categoria": "BIE",
      "keywords_primarios": [
        "psicologo",
        "psicologia",
        "salud mental"
      ],
      "keywords_secundarios": [
        "gratuito",
        "cita",
        "confidencial"
      ],
      "sinonimos": [
        "atencion psicologica"
      ],
      "frases_trigger": [
        "psicologo unsaac",
        "cita psicologo",
        "salud mental universidad"
      ]
    },
    {
      "intent_id": "consulta_bie_beca_alimentacion",
      "categoria": "BIE",
      "keywords_primarios": [
        "beca alimentacion",
        "comedor gratis"
      ],
      "keywords_secundarios": [
        "declaracion jurada",
        "solicitud bienestar"
      ],
      "sinonimos": [
        "beca comida"
      ],
      "frases_trigger": [
        "beca alimentacion",
        "comedor gratis",
        "como solicitar beca comida"
      ]
    },
    {
      "intent_id": "consulta_bie_apoyo_economico",
      "categoria": "BIE",
      "keywords_primarios": [
        "apoyo economico",
        "beca",
        "subvencion"
      ],
      "keywords_secundarios": [
        "beca 18",
        "minedu",
        "exoneracion"
      ],
      "sinonimos": [
        "ayuda economica"
      ],
      "frases_trigger": [
        "apoyo economico universidad",
        "beca 18",
        "ayuda alumnos pobres"
      ]
    },
    {
      "intent_id": "consulta_bie_seguro_estudiantil",
      "categoria": "BIE",
      "keywords_primarios": [
        "seguro estudiantil",
        "seguro accidentes"
      ],
      "keywords_secundarios": [
        "carne vigente",
        "cobertura",
        "topico"
      ],
      "sinonimos": [
        "seguro universitario"
      ],
      "frases_trigger": [
        "seguro estudiantil",
        "que cubre seguro",
        "seguro accidentes campus"
      ]
    },
    {
      "intent_id": "consulta_bie_deportes_cultura",
      "categoria": "BIE",
      "keywords_primarios": [
        "deportes",
        "actividades culturales",
        "futbol"
      ],
      "keywords_secundarios": [
        "basquet",
        "danza",
        "teatro",
        "inscripcion"
      ],
      "sinonimos": [
        "actividades extracurriculares"
      ],
      "frases_trigger": [
        "deportes universidad",
        "actividades culturales unsaac",
        "hacer deporte campus"
      ]
    },
    {
      "intent_id": "consulta_bie_residencia_universitaria",
      "categoria": "BIE",
      "keywords_primarios": [
        "residencia",
        "albergue",
        "dormitorio"
      ],
      "keywords_secundarios": [
        "foraneos",
        "provincias",
        "socioeconomico"
      ],
      "sinonimos": [
        "alojamiento universitario"
      ],
      "frases_trigger": [
        "residencia universitaria",
        "albergue estudiantes",
        "dormitorio campus"
      ]
    },
    {
      "intent_id": "consulta_bie_certificado_socioeconomico",
      "categoria": "BIE",
      "keywords_primarios": [
        "certificado socioeconomico",
        "constancia socioeconomica"
      ],
      "keywords_secundarios": [
        "servicio social",
        "declaracion jurada"
      ],
      "sinonimos": [
        "estudio socioeconomico"
      ],
      "frases_trigger": [
        "como solicito socioeconomico",
        "donde tramito constancia socioeconomica",
        "estudio socioeconomico unsaac"
      ]
    },
    {
      "intent_id": "consulta_bie_comedor_acceso",
      "categoria": "BIE",
      "keywords_primarios": [
        "comedor",
        "precio comedor",
        "tickets"
      ],
      "keywords_secundarios": [
        "subsidiado",
        "beca",
        "todos estudiantes"
      ],
      "sinonimos": [
        "comer campus"
      ],
      "frases_trigger": [
        "comedor universitario",
        "precio menu",
        "comedor abierto todos"
      ]
    },
    {
      "intent_id": "consulta_bie_contacto_oficina",
      "categoria": "BIE",
      "keywords_primarios": [
        "contacto bienestar",
        "correo bienestar",
        "telefono"
      ],
      "keywords_secundarios": [
        "perayoc",
        "anexo",
        "oficina"
      ],
      "sinonimos": [
        "donde bienestar"
      ],
      "frases_trigger": [
        "numero bienestar",
        "correo bienestar",
        "donde oficina bienestar"
      ]
    },
    {
      "intent_id": "consulta_mov_programas_disponibles",
      "categoria": "MOV",
      "keywords_primarios": [
        "movilidad",
        "intercambio",
        "convenios internacionales"
      ],
      "keywords_secundarios": [
        "pame",
        "udual",
        "pronabec"
      ],
      "sinonimos": [
        "programas intercambio"
      ],
      "frases_trigger": [
        "que programas movilidad",
        "intercambios estudiantiles",
        "convenios intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_requisitos_intercambio",
      "categoria": "MOV",
      "keywords_primarios": [
        "requisitos intercambio",
        "postular movilidad"
      ],
      "keywords_secundarios": [
        "promedio 14",
        "idioma",
        "carta motivacion"
      ],
      "sinonimos": [
        "condiciones intercambio"
      ],
      "frases_trigger": [
        "que necesito intercambio",
        "requisitos movilidad",
        "como postulo intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_convalidacion_cursos",
      "categoria": "CUR",
      "keywords_primarios": [
        "convalidacion",
        "homologacion",
        "reconocer cursos"
      ],
      "keywords_secundarios": [
        "silabos",
        "equivalencia",
        "creditos"
      ],
      "sinonimos": [
        "validar cursos intercambio"
      ],
      "frases_trigger": [
        "convalidan cursos intercambio",
        "reconocen materias intercambio",
        "creditos intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_apoyo_economico",
      "categoria": "MOV",
      "keywords_primarios": [
        "beca intercambio",
        "financiamiento",
        "apoyo"
      ],
      "keywords_secundarios": [
        "pronabec",
        "fundacion carolina",
        "pasajes"
      ],
      "sinonimos": [
        "ayuda intercambio"
      ],
      "frases_trigger": [
        "beca intercambio",
        "financia intercambio",
        "ayuda pagar intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_estudiantes_entrantes",
      "categoria": "MOV",
      "keywords_primarios": [
        "entrantes",
        "incoming",
        "extranjeros"
      ],
      "keywords_secundarios": [
        "admision",
        "bilateral",
        "matriculan"
      ],
      "sinonimos": [
        "estudiantes intercambio"
      ],
      "frases_trigger": [
        "unsaac recibe extranjeros",
        "estudiantes intercambio entran",
        "movilidad entrante"
      ]
    },
    {
      "intent_id": "consulta_mov_convocatorias",
      "categoria": "MOV",
      "keywords_primarios": [
        "convocatorias",
        "publicacion",
        "cuando convocatoria"
      ],
      "keywords_secundarios": [
        "serunsa",
        "redes sociales",
        "relaciones internacionales"
      ],
      "sinonimos": [
        "avisos movilidad"
      ],
      "frases_trigger": [
        "donde convocatorias movilidad",
        "como entero intercambio",
        "cuando hay intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_documentos_postulacion",
      "categoria": "MOV",
      "keywords_primarios": [
        "documentos",
        "expediente",
        "papeles"
      ],
      "keywords_secundarios": [
        "curriculum",
        "carta motivacion",
        "recomendacion",
        "certificado notas"
      ],
      "sinonimos": [
        "postulacion intercambio"
      ],
      "frases_trigger": [
        "que documentos intercambio",
        "expediente movilidad",
        "papeles piden intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_impacto_titulacion",
      "categoria": "MOV",
      "keywords_primarios": [
        "retrasar",
        "impacto titulacion",
        "perder semestre"
      ],
      "keywords_secundarios": [
        "homologacion",
        "planificar"
      ],
      "sinonimos": [
        "intercambio retrasa carrera"
      ],
      "frases_trigger": [
        "intercambio retrasa",
        "titulacion intercambio",
        "perdo semestre intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_requisito_idioma",
      "categoria": "MOV",
      "keywords_primarios": [
        "idioma",
        "ingles",
        "nivel",
        "toefl"
      ],
      "keywords_secundarios": [
        "ielts",
        "portugues",
        "frances",
        "nivel intermedio"
      ],
      "sinonimos": [
        "certificado idioma"
      ],
      "frases_trigger": [
        "que idioma intercambio",
        "nivel ingles movilidad",
        "idioma obligatorio intercambio"
      ]
    },
    {
      "intent_id": "consulta_mov_universidades_convenio",
      "categoria": "MOV",
      "keywords_primarios": [
        "universidades convenio",
        "alianzas internacionales"
      ],
      "keywords_secundarios": [
        "salamanca",
        "unam",
        "brasil",
        "argentina"
      ],
      "sinonimos": [
        "convenios internacionales"
      ],
      "frases_trigger": [
        "convenios universidades",
        "alianzas unsaac",
        "universidades extranjeras convenio"
      ]
    },
    {
      "intent_id": "consulta_mat_fechas_matricula",
      "categoria": "MAT",
      "keywords_primarios": [
        "fechas matricula",
        "cuando matricula",
        "periodos"
      ],
      "keywords_secundarios": [
        "cronograma",
        "draa",
        "marzo",
        "agosto"
      ],
      "sinonimos": [
        "inicio matricula"
      ],
      "frases_trigger": [
        "cuando es matricula",
        "fechas matricula",
        "periodos matricula"
      ]
    },
    {
      "intent_id": "consulta_mat_proceso_serunsa",
      "categoria": "MAT",
      "keywords_primarios": [
        "serunsa",
        "matricula online",
        "pasos matricula"
      ],
      "keywords_secundarios": [
        "usuario",
        "contrasena",
        "comprobante"
      ],
      "sinonimos": [
        "como matricularse"
      ],
      "frases_trigger": [
        "como matriculo serunsa",
        "pasos matricula",
        "matricula en linea"
      ]
    },
    {
      "intent_id": "consulta_mat_requisitos",
      "categoria": "MAT",
      "keywords_primarios": [
        "requisitos matricula",
        "que necesito matricularme"
      ],
      "keywords_secundarios": [
        "deudas",
        "carne",
        "habilitacion"
      ],
      "sinonimos": [
        "condiciones matricula"
      ],
      "frases_trigger": [
        "que necesito matricularme",
        "requisitos matricula",
        "condiciones matricula"
      ]
    },
    {
      "intent_id": "consulta_mat_modalidad_presencial",
      "categoria": "MAT",
      "keywords_primarios": [
        "presencial",
        "en persona",
        "sin internet"
      ],
      "keywords_secundarios": [
        "draa",
        "modulos",
        "dificultades"
      ],
      "sinonimos": [
        "matricula presencial"
      ],
      "frases_trigger": [
        "matricula presencial",
        "en persona matricula",
        "sin internet matricularme"
      ]
    },
    {
      "intent_id": "consulta_mat_fuera_de_plazo",
      "categoria": "MAT",
      "keywords_primarios": [
        "fuera de plazo",
        "extemporanea",
        "vencio plazo"
      ],
      "keywords_secundarios": [
        "cinco dias",
        "solicitud",
        "vacantes"
      ],
      "sinonimos": [
        "matricula tarde"
      ],
      "frases_trigger": [
        "no pude matricularme",
        "matricula extemporanea",
        "vencio plazo matricula"
      ]
    },
    {
      "intent_id": "consulta_mat_creditos_maximo",
      "categoria": "MAT",
      "keywords_primarios": [
        "creditos maximo",
        "24 creditos",
        "carga maxima"
      ],
      "keywords_secundarios": [
        "sobrecarga",
        "minimo 12",
        "regular"
      ],
      "sinonimos": [
        "limite creditos"
      ],
      "frases_trigger": [
        "creditos maximo semestre",
        "cuantos cursos llevar",
        "limite materias semestre"
      ]
    },
    {
      "intent_id": "consulta_mat_retiro_curso",
      "categoria": "MAT",
      "keywords_primarios": [
        "retirar curso",
        "baja curso",
        "cancelar curso"
      ],
      "keywords_secundarios": [
        "cuarta semana",
        "registro negativo"
      ],
      "sinonimos": [
        "dar de baja"
      ],
      "frases_trigger": [
        "puedo retirar curso",
        "baja curso matriculado",
        "cuanto tiempo retirar curso"
      ]
    },
    {
      "intent_id": "consulta_mat_condicional",
      "categoria": "MAT",
      "keywords_primarios": [
        "condicional",
        "matricula condicional",
        "bajo promedio"
      ],
      "keywords_secundarios": [
        "dos oportunidades",
        "acta compromiso",
        "seguimiento"
      ],
      "sinonimos": [
        "condicion academica"
      ],
      "frases_trigger": [
        "que es matricula condicional",
        "cuando aplica condicional",
        "me ponen condicional"
      ]
    },
    {
      "intent_id": "consulta_mat_costo",
      "categoria": "MAT",
      "keywords_primarios": [
        "costo matricula",
        "precio matricula",
        "tasa"
      ],
      "keywords_secundarios": [
        "socioeconomico",
        "consejo universitario"
      ],
      "sinonimos": [
        "cuanto cuesta matricula"
      ],
      "frases_trigger": [
        "cuanto cuesta matricula",
        "precio matricula unsaac",
        "tasa matricula"
      ]
    },
    {
      "intent_id": "consulta_mat_seguro_activacion",
      "categoria": "MAT",
      "keywords_primarios": [
        "seguro activacion",
        "seguro matricula"
      ],
      "keywords_secundarios": [
        "automatico",
        "primer dia",
        "carnet"
      ],
      "sinonimos": [
        "seguro al matricularse"
      ],
      "frases_trigger": [
        "seguro activa matricula",
        "cuando activa seguro",
        "matricula incluye seguro"
      ]
    },
    {
      "intent_id": "consulta_tit_modalidades",
      "categoria": "TIT",
      "keywords_primarios": [
        "modalidades titulacion",
        "como titular",
        "opciones titulo"
      ],
      "keywords_secundarios": [
        "tesis",
        "suficiencia",
        "bachiller",
        "examen"
      ],
      "sinonimos": [
        "formas titularse"
      ],
      "frases_trigger": [
        "modalidades titulacion",
        "como me titulo",
        "opciones obtener titulo"
      ]
    },
    {
      "intent_id": "consulta_tit_requisitos_inicio",
      "categoria": "TIT",
      "keywords_primarios": [
        "requisitos titulacion",
        "que necesito titular"
      ],
      "keywords_secundarios": [
        "220 creditos",
        "bachiller",
        "practicas",
        "expediente"
      ],
      "sinonimos": [
        "condiciones titulacion"
      ],
      "frases_trigger": [
        "que necesito titularme",
        "requisitos titulacion",
        "documentos titulo"
      ]
    },
    {
      "intent_id": "consulta_tit_grado_bachiller",
      "categoria": "TIT",
      "keywords_primarios": [
        "bachiller",
        "grado bachiller",
        "bachillerato"
      ],
      "keywords_secundarios": [
        "automatico",
        "ley 30220",
        "trabajo investigacion"
      ],
      "sinonimos": [
        "obtener bachiller"
      ],
      "frases_trigger": [
        "como obtengo bachiller",
        "que necesito bachillerato",
        "bachiller automatico"
      ]
    },
    {
      "intent_id": "consulta_tit_plazo_titulacion",
      "categoria": "TIT",
      "keywords_primarios": [
        "plazo titulacion",
        "tiempo titularse",
        "limite"
      ],
      "keywords_secundarios": [
        "caducan creditos",
        "recomendable"
      ],
      "sinonimos": [
        "cuanto tiempo para titularse"
      ],
      "frases_trigger": [
        "plazo para titularme",
        "tiempo tengo para titulo",
        "caducan creditos"
      ]
    },
    {
      "intent_id": "consulta_tit_suficiencia_profesional",
      "categoria": "TIT",
      "keywords_primarios": [
        "suficiencia profesional",
        "titulacion suficiencia"
      ],
      "keywords_secundarios": [
        "experiencia laboral",
        "informe",
        "proyecto"
      ],
      "sinonimos": [
        "modalidad suficiencia"
      ],
      "frases_trigger": [
        "que es suficiencia profesional",
        "como titulo suficiencia",
        "informe suficiencia"
      ]
    },
    {
      "intent_id": "consulta_tit_eleccion_asesor",
      "categoria": "TIT",
      "keywords_primarios": [
        "asesor tesis",
        "elegir asesor",
        "docente asesor"
      ],
      "keywords_secundarios": [
        "disponibilidad",
        "perfil",
        "cinco tesistas"
      ],
      "sinonimos": [
        "como escoger asesor"
      ],
      "frases_trigger": [
        "como escojo asesor tesis",
        "quien puede ser asesor",
        "elegir asesor investigacion"
      ]
    },
    {
      "intent_id": "consulta_tit_duracion_proceso",
      "categoria": "TIT",
      "keywords_primarios": [
        "cuanto dura titulo",
        "tiempo titulacion",
        "meses"
      ],
      "keywords_secundarios": [
        "12 meses",
        "24 meses",
        "sunedu"
      ],
      "sinonimos": [
        "duracion titulacion"
      ],
      "frases_trigger": [
        "cuanto dura titularse",
        "tiempo sacar titulo",
        "tramite titulacion rapido"
      ]
    },
    {
      "intent_id": "consulta_tit_jurado_sustentacion",
      "categoria": "TIT",
      "keywords_primarios": [
        "jurado",
        "tres docentes",
        "sustentacion"
      ],
      "keywords_secundarios": [
        "presidente",
        "miembro",
        "nota promedio"
      ],
      "sinonimos": [
        "conformacion jurado"
      ],
      "frases_trigger": [
        "quienes jurado tesis",
        "como es jurado sustentacion",
        "cuantos evaluan sustentacion"
      ]
    },
    {
      "intent_id": "consulta_tit_desaprobacion_sustentacion",
      "categoria": "TIT",
      "keywords_primarios": [
        "desapruebo sustentacion",
        "jale sustentacion"
      ],
      "keywords_secundarios": [
        "30 dias",
        "dos oportunidades",
        "observaciones"
      ],
      "sinonimos": [
        "segunda sustentacion"
      ],
      "frases_trigger": [
        "que pasa desapruebo sustentacion",
        "repetir sustentacion",
        "segunda oportunidad defensa"
      ]
    },
    {
      "intent_id": "consulta_tit_registro_sunedu",
      "categoria": "TIT",
      "keywords_primarios": [
        "sunedu",
        "registro titulo",
        "validez"
      ],
      "keywords_secundarios": [
        "codigo unico",
        "nacional",
        "verificar"
      ],
      "sinonimos": [
        "inscribir titulo"
      ],
      "frases_trigger": [
        "donde registro titulo",
        "sunedu titulo",
        "validez titulo nacional"
      ]
    },
    {
      "intent_id": "consulta_ser_biblioteca",
      "categoria": "SER",
      "keywords_primarios": [
        "biblioteca",
        "repositorio",
        "ebsco"
      ],
      "keywords_secundarios": [
        "scopus",
        "digital",
        "presencial",
        "lunes sabado"
      ],
      "sinonimos": [
        "recursos biblioteca"
      ],
      "frases_trigger": [
        "como accedo biblioteca",
        "usar biblioteca unsaac",
        "biblioteca virtual"
      ]
    },
    {
      "intent_id": "consulta_ser_correo_institucional",
      "categoria": "TIT",
      "keywords_primarios": [
        "correo institucional",
        "email unsaac",
        "activar correo"
      ],
      "keywords_secundarios": [
        "dti",
        "google workspace",
        "microsoft"
      ],
      "sinonimos": [
        "email universitario"
      ],
      "frases_trigger": [
        "como obtengo correo",
        "activar email universitario",
        "correo de la unsaac"
      ]
    },
    {
      "intent_id": "consulta_ser_certificados_constancias",
      "categoria": "SER",
      "keywords_primarios": [
        "certificados",
        "constancias",
        "tramitar"
      ],
      "keywords_secundarios": [
        "draa",
        "serunsa",
        "dias habiles"
      ],
      "sinonimos": [
        "documentos academicos"
      ],
      "frases_trigger": [
        "donde tramito certificados",
        "como pido constancia",
        "certificados notas"
      ]
    },
    {
      "intent_id": "consulta_ser_aula_virtual",
      "categoria": "SER",
      "keywords_primarios": [
        "aula virtual",
        "moodle",
        "campus virtual"
      ],
      "keywords_secundarios": [
        "serunsa",
        "plataforma",
        "elearning"
      ],
      "sinonimos": [
        "entorno virtual"
      ],
      "frases_trigger": [
        "como accedo aula virtual",
        "donde esta campus virtual",
        "plataforma elearning unsaac"
      ]
    },
    {
      "intent_id": "consulta_ser_laboratorios_computo",
      "categoria": "SER",
      "keywords_primarios": [
        "laboratorios",
        "computo",
        "computadoras"
      ],
      "keywords_secundarios": [
        "visual studio",
        "matlab",
        "cisco",
        "cuatro"
      ],
      "sinonimos": [
        "laboratorios informatica"
      ],
      "frases_trigger": [
        "laboratorios computo",
        "computadoras campus",
        "laboratorios epiis"
      ]
    },
    {
      "intent_id": "consulta_ser_carta_presentacion",
      "categoria": "SER",
      "keywords_primarios": [
        "carta presentacion evento",
        "concurso",
        "solicitud"
      ],
      "keywords_secundarios": [
        "tres dias",
        "visto bueno",
        "secretaria"
      ],
      "sinonimos": [
        "carta evento concurso"
      ],
      "frases_trigger": [
        "como pido carta concurso",
        "carta presentacion evento",
        "escuela emite carta"
      ]
    },
    {
      "intent_id": "consulta_ser_impresion_campus",
      "categoria": "SER",
      "keywords_primarios": [
        "impresion",
        "fotocopiado",
        "imprimir"
      ],
      "keywords_secundarios": [
        "campus",
        "pabellones",
        "carne"
      ],
      "sinonimos": [
        "copias campus"
      ],
      "frases_trigger": [
        "donde imprimo campus",
        "fotocopiadoras universidad",
        "servicio impresion"
      ]
    },
    {
      "intent_id": "consulta_ser_record_notas",
      "categoria": "SER",
      "keywords_primarios": [
        "record notas",
        "notas certificadas",
        "notas oficial"
      ],
      "keywords_secundarios": [
        "draa",
        "tupa",
        "sello",
        "firma"
      ],
      "sinonimos": [
        "certificado notas"
      ],
      "frases_trigger": [
        "donde saco record notas",
        "notas certificadas",
        "tramitar record notas"
      ]
    },
    {
      "intent_id": "consulta_ser_tupa",
      "categoria": "SER",
      "keywords_primarios": [
        "tupa",
        "texto unico",
        "tramites costos"
      ],
      "keywords_secundarios": [
        "transparencia",
        "secretaria general",
        "procedimientos"
      ],
      "sinonimos": [
        "que es tupa"
      ],
      "frases_trigger": [
        "que es tupa",
        "costos tramites",
        "donde consulto tupa"
      ]
    },
    {
      "intent_id": "consulta_ser_contacto_secretaria",
      "categoria": "SER",
      "keywords_primarios": [
        "secretaria epiis",
        "correo secretaria",
        "telefono escuela"
      ],
      "keywords_secundarios": [
        "perayoc",
        "horario",
        "lunes viernes"
      ],
      "sinonimos": [
        "contacto secretaria"
      ],
      "frases_trigger": [
        "como contacto secretaria",
        "correo secretaria sistemas",
        "donde esta secretaria epiis"
      ]
    }
  ]
}

FAQ_INDEX_DATA = {
  "version": "1.1.0",
  "total_preguntas": 100,
  "total_categorias": 10,
  "indice": {
    "TUT": {
      "nombre": "Proceso de Tutorías",
      "preguntas_count": 10
    },
    "TTUT": {
      "nombre": "Tipos de Tutoría",
      "preguntas_count": 10
    },
    "CUR": {
      "nombre": "Cursos y Semestres",
      "preguntas_count": 10
    },
    "ESP": {
      "nombre": "Especialidades",
      "preguntas_count": 10
    },
    "PPP": {
      "nombre": "Prácticas Pre-Profesionales",
      "preguntas_count": 10
    },
    "BIE": {
      "nombre": "Bienestar Universitario",
      "preguntas_count": 10
    },
    "MOV": {
      "nombre": "Movilidad Estudiantil",
      "preguntas_count": 10
    },
    "MAT": {
      "nombre": "Matrícula",
      "preguntas_count": 10
    },
    "TIT": {
      "nombre": "Titulación",
      "preguntas_count": 10
    },
    "SER": {
      "nombre": "Servicios Universitarios",
      "preguntas_count": 10
    }
  },
  "busqueda_rapida": {
    "consulta_tutoria_definicion": "TUT-001",
    "consulta_tutoria_frecuencia": "TUT-002",
    "consulta_tutoria_contacto_tutor": "TUT-003",
    "consulta_tutoria_temas": "TUT-004",
    "consulta_tutoria_obligatoriedad": "TUT-005",
    "consulta_tutoria_registro_asistencia": "TUT-006",
    "consulta_tutoria_cambio_tutor": "TUT-007",
    "consulta_tutoria_reglamento": "TUT-008",
    "consulta_tutoria_materiales_sesion": "TUT-009",
    "consulta_tutoria_confidencialidad": "TUT-010",
    "consulta_tipos_tutoria_general": "TTUT-001",
    "consulta_tipos_tutoria_individual": "TTUT-002",
    "consulta_tipos_tutoria_grupal": "TTUT-003",
    "consulta_tipos_tutoria_pares": "TTUT-004",
    "consulta_tipos_tutoria_virtual": "TTUT-005",
    "consulta_tipos_tutoria_vs_psicologia": "TTUT-006",
    "consulta_tipos_tutoria_ingresantes": "TTUT-007",
    "consulta_tipos_tutoria_riesgo_academico": "TTUT-008",
    "consulta_tipos_tutoria_vs_asesoria_tesis": "TTUT-009",
    "consulta_tipos_tutoria_vocacional": "TTUT-010",
    "consulta_cursos_duracion_carrera": "CUR-001",
    "consulta_cursos_total_creditos": "CUR-002",
    "consulta_cursos_primer_semestre": "CUR-003",
    "consulta_cursos_adelanto_semestres": "CUR-004",
    "consulta_cursos_desaprobacion": "CUR-005",
    "consulta_cursos_areas_plan_estudios": "CUR-006",
    "consulta_cursos_electivos": "CUR-007",
    "consulta_cursos_horario_clases": "CUR-008",
    "consulta_cursos_creditos_definicion": "CUR-009",
    "consulta_cursos_promedio_ponderado": "CUR-010",
    "consulta_esp_listado_especialidades": "ESP-001",
    "consulta_esp_cuando_elegir": "ESP-002",
    "consulta_esp_demanda_laboral": "ESP-003",
    "consulta_esp_cursos_ingenieria_software": "ESP-004",
    "consulta_esp_cambio_especialidad": "ESP-005",
    "consulta_esp_cursos_redes_telecom": "ESP-006",
    "consulta_esp_ia_ciencia_datos": "ESP-007",
    "consulta_esp_seguridad_certificaciones": "ESP-008",
    "consulta_esp_convenios_practicas": "ESP-009",
    "consulta_esp_electivos_cruzados": "ESP-010",
    "consulta_ppp_requisitos_inicio": "PPP-001",
    "consulta_ppp_horas_acumuladas": "PPP-002",
    "consulta_ppp_empresa_convenio": "PPP-003",
    "consulta_ppp_carta_presentacion": "PPP-004",
    "consulta_ppp_informe_final": "PPP-005",
    "consulta_ppp_remuneracion": "PPP-006",
    "consulta_ppp_sector_publico": "PPP-007",
    "consulta_ppp_supervision_universidad": "PPP-008",
    "consulta_ppp_trabajo_actual": "PPP-009",
    "consulta_ppp_requisito_titulacion": "PPP-010",
    "consulta_bie_servicios_generales": "BIE-001",
    "consulta_bie_atencion_psicologica": "BIE-002",
    "consulta_bie_beca_alimentacion": "BIE-003",
    "consulta_bie_apoyo_economico": "BIE-004",
    "consulta_bie_seguro_estudiantil": "BIE-005",
    "consulta_bie_deportes_cultura": "BIE-006",
    "consulta_bie_residencia_universitaria": "BIE-007",
    "consulta_bie_certificado_socioeconomico": "BIE-008",
    "consulta_bie_comedor_acceso": "BIE-009",
    "consulta_bie_contacto_oficina": "BIE-010",
    "consulta_mov_programas_disponibles": "MOV-001",
    "consulta_mov_requisitos_intercambio": "MOV-002",
    "consulta_mov_convalidacion_cursos": "MOV-003",
    "consulta_mov_apoyo_economico": "MOV-004",
    "consulta_mov_estudiantes_entrantes": "MOV-005",
    "consulta_mov_convocatorias": "MOV-006",
    "consulta_mov_documentos_postulacion": "MOV-007",
    "consulta_mov_impacto_titulacion": "MOV-008",
    "consulta_mov_requisito_idioma": "MOV-009",
    "consulta_mov_universidades_convenio": "MOV-010",
    "consulta_mat_fechas_matricula": "MAT-001",
    "consulta_mat_proceso_serunsa": "MAT-002",
    "consulta_mat_requisitos": "MAT-003",
    "consulta_mat_modalidad_presencial": "MAT-004",
    "consulta_mat_fuera_de_plazo": "MAT-005",
    "consulta_mat_creditos_maximo": "MAT-006",
    "consulta_mat_retiro_curso": "MAT-007",
    "consulta_mat_condicional": "MAT-008",
    "consulta_mat_costo": "MAT-009",
    "consulta_mat_seguro_activacion": "MAT-010",
    "consulta_tit_modalidades": "TIT-001",
    "consulta_tit_requisitos_inicio": "TIT-002",
    "consulta_tit_grado_bachiller": "TIT-003",
    "consulta_tit_plazo_titulacion": "TIT-004",
    "consulta_tit_suficiencia_profesional": "TIT-005",
    "consulta_tit_eleccion_asesor": "TIT-006",
    "consulta_tit_duracion_proceso": "TIT-007",
    "consulta_tit_jurado_sustentacion": "TIT-008",
    "consulta_tit_desaprobacion_sustentacion": "TIT-009",
    "consulta_tit_registro_sunedu": "TIT-010",
    "consulta_ser_biblioteca": "SER-001",
    "consulta_ser_correo_institucional": "SER-002",
    "consulta_ser_certificados_constancias": "SER-003",
    "consulta_ser_aula_virtual": "SER-004",
    "consulta_ser_laboratorios_computo": "SER-005",
    "consulta_ser_carta_presentacion": "SER-006",
    "consulta_ser_impresion_campus": "SER-007",
    "consulta_ser_record_notas": "SER-008",
    "consulta_ser_tupa": "SER-009",
    "consulta_ser_contacto_secretaria": "SER-010"
  }
}

(BASE / "corpus" / "corpus_general.json").write_text(
    json.dumps(CORPUS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")

(BASE / "knowledge_base" / "intents.json").write_text(
    json.dumps(INTENTS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")

(BASE / "knowledge_base" / "keywords.json").write_text(
    json.dumps(KEYWORDS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")

(BASE / "knowledge_base" / "faq_index.json").write_text(
    json.dumps(FAQ_INDEX_DATA, ensure_ascii=False, indent=2), encoding="utf-8")

print("✓ corpus_general.json   →", len(CORPUS_DATA), "entradas")
print("✓ intents.json          →", len(INTENTS_DATA['intents']), "intents")
print("✓ keywords.json         →", len(KEYWORDS_DATA['keywords']), "entradas keyword")
print("✓ faq_index.json        → mapa de", len(FAQ_INDEX_DATA['busqueda_rapida']), "intents")


✓ corpus_general.json   → 100 entradas
✓ intents.json          → 100 intents
✓ keywords.json         → 100 entradas keyword
✓ faq_index.json        → mapa de 100 intents


In [16]:
# ═══════════════════════════════════════════════════
# CELDA 3 — Generar knowledge_files JSON
# ═══════════════════════════════════════════════════
true = True
TUTORIAS_DATA = {
  "version": "1.0.0",
  "fuente_normativa": "Reglamento de Tutoría Académica UNSAAC",
  "resolucion": "N° CU-0220-2017-UNSAAC",
  "fecha_aprobacion": "2017-05-24",
  "base_legal": {
    "ley_universitaria": "Ley N° 30220, Art. 87.5",
    "estatuto_unsaac": "Art. 195.5"
  },
  "definicion": {
    "art": "Art. 3°",
    "texto": "La Tutoría Académica es un proceso permanente de acompañamiento durante la formación de los estudiantes, que se concreta mediante la atención personalizada o grupal por parte de docentes, orientándolos y proporcionándoles seguimiento en los aspectos psicosociales, cognitivos y afectivos del aprendizaje."
  },
  "fines": {
    "art": "Art. 4°",
    "descripcion": "Constituirse como medio para hacer auténticos los fines de la Ley Universitaria, asegurar la excelencia integral y sostenible en la formación profesional.",
    "nota_especial": "Es obligatorio asignar un tutor al estudiante con matrícula condicionada."
  },
  "sujetos": {
    "art": "Art. 6°",
    "tutor": "Docente universitario con régimen de tiempo completo o dedicación exclusiva, acreditado para promover la formación integral.",
    "tutorado": "El estudiante universitario."
  },
  "dimensiones": {
    "art": "Art. 7°",
    "academica": [
      "Conocer las exigencias de las diversas opciones académicas",
      "Aprender habilidades de estudio eficaces",
      "Fomentar habilidades de pensamiento crítico",
      "Identificar los estilos de aprendizaje individuales",
      "Promover habilidades de toma de decisiones"
    ],
    "personal": [
      "Fomentar el conocimiento y aceptación de sí mismo",
      "Desarrollar el sentido de la responsabilidad personal",
      "Promover habilidades interpersonales y de comunicación",
      "Promover el trabajo en grupo",
      "Fomentar la comprensión y el respeto hacia los demás"
    ],
    "profesional": [
      "Conocer características, intereses, aptitudes y habilidades propias",
      "Fomentar el conocimiento del mundo del trabajo",
      "Comprender la relación entre rendimiento académico y elecciones de futuro",
      "Desarrollar actitud positiva hacia el mundo del trabajo",
      "Examinar la influencia de los cambios en el mundo del trabajo"
    ]
  },
  "periodicidad": {
    "art": "Art. 12°",
    "momentos_clave_semestre": [
      "Al comienzo del curso académico",
      "Después de la primera evaluación parcial",
      "Una semana antes de la finalización del semestre"
    ]
  },
  "asignacion_tutores": {
    "art": "Art. 14°",
    "desde": "Primer semestre de estudios",
    "hasta": "Culminación del plan curricular",
    "maximo_tutorados": 25,
    "cambio_tutor": "El estudiante puede solicitar cambio razonadamente al Comité Tutorial; resuelve el Vicerrectorado Académico en última instancia."
  },
  "confidencialidad": {
    "art": "Art. 15°",
    "deber": "Los tutores tienen deber de confidencialidad sobre la información de los estudiantes.",
    "excepcion": "En casos graves, puede informar a padres/representantes solo en circunstancias excepcionales (Art. 13.8)."
  }
}
PRACTICAS_DATA = {
  "version": "1.0.0",
  "categoria_id": "PPP",
  "nombre": "Prácticas Pre-Profesionales",
  "fuentes_normativas": [
    "Reglamento de Prácticas Pre-Profesionales UNSAAC",
    "Ley N° 28518 - Ley de Modalidades Formativas Laborales",
    "Decreto Legislativo N° 1401"
  ],
  "requisitos_inicio": {
    "condiciones": [
      "Haber cursado y aprobado el IX semestre",
      "Tener un promedio ponderado mínimo de 11",
      "Contar con carta de presentación de la Dirección de la Escuela",
      "Tener acuerdo firmado con empresa o institución convenio",
      "Estar matriculado en el curso de Prácticas Pre-Profesionales"
    ]
  },
  "duracion": {
    "horas_minimas": 300,
    "equivalencia": "Aproximadamente tres meses a tiempo completo"
  },
  "empresa_centro_practicas": {
    "prioridad": "Empresas con convenio vigente UNSAAC",
    "sin_convenio": "Posible con carta de autorización de la Dirección de Escuela",
    "sector_publico": {
      "permitido": true,
      "norma": "Decreto Legislativo N° 1401"
    }
  },
  "carta_presentacion": {
    "emisor": "Dirección de la Escuela Profesional",
    "plazo_emision": "Tres días hábiles"
  },
  "remuneracion": {
    "norma": "Ley N° 28518",
    "subvencion_minima": "50% de la Remuneración Mínima Vital vigente",
    "seguro_salud": true
  },
  "supervision_universitaria": {
    "responsable": "Asesor de Prácticas (docente designado)"
  },
  "relacion_titulacion": {
    "es_requisito": true,
    "descripcion": "La aprobación del curso de PPP y presentación del informe son requisitos obligatorios para iniciar el trámite de titulación."
  }
}
SERVICIOS_DATA = {
  "version": "1.0.0",
  "institucion": "UNSAAC - EPIIS",
  "modulos": {
    "bienestar_universitario": {
      "categoria_id": "BIE",
      "dependencia": "Dirección de Bienestar y Responsabilidad Social UNSAAC",
      "ubicacion": "Campus Universitario de Perayoc",
      "servicios": [
        "Atención médica y odontológica",
        "Orientación psicológica",
        "Beca de alimentación",
        "Apoyo económico y Beca 18",
        "Seguro estudiantil de accidentes",
        "Actividades deportivas y culturales",
        "Residencia universitaria",
        "Certificado socioeconómico",
        "Comedor universitario"
      ]
    },
    "movilidad_estudiantil": {
      "categoria_id": "MOV",
      "dependencia": "Oficina de Relaciones Internacionales UNSAAC",
      "programas": [
        "PAME-UDUAL",
        "Convenios bilaterales (Salamanca, UNAM, Brasil, Argentina)",
        "Movilidad Nacional",
        "Becas PRONABEC / Fundación Carolina"
      ],
      "requisitos_generales": [
        "Matriculado desde IV semestre",
        "Promedio mínimo 14/20",
        "Nivel intermedio del idioma destino",
        "Sin sanciones disciplinarias",
        "Carta de motivación y plan de estudios equivalente"
      ]
    },
    "servicios_academicos": {
      "categoria_id": "SER",
      "servicios": [
        {
          "nombre": "Biblioteca Central",
          "acceso_virtual": [
            "repositorio.unsaac.edu.pe",
            "EBSCO",
            "Scopus"
          ]
        },
        {
          "nombre": "Correo Institucional",
          "formato": "nombre.apellido@unsaac.edu.pe",
          "tramite": "DTI con carné universitario"
        },
        {
          "nombre": "Aula Virtual Moodle",
          "url": "aulavirtual.unsaac.edu.pe"
        },
        {
          "nombre": "Portal SERUNSA",
          "url": "serunsa.unsaac.edu.pe",
          "funciones": [
            "Matrícula",
            "Horarios",
            "Notas",
            "Constancias"
          ]
        },
        {
          "nombre": "Laboratorios EPIIS",
          "cantidad": 4,
          "software": [
            "Visual Studio",
            "Eclipse",
            "MATLAB",
            "Cisco Packet Tracer"
          ]
        },
        {
          "nombre": "TUPA",
          "url": "Portal UNSAAC sección Transparencia"
        }
      ]
    }
  }
}

(BASE / "knowledge_files" / "tutorias.json").write_text(
    json.dumps(TUTORIAS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")
(BASE / "knowledge_files" / "practicas.json").write_text(
    json.dumps(PRACTICAS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")
(BASE / "knowledge_files" / "servicios.json").write_text(
    json.dumps(SERVICIOS_DATA, ensure_ascii=False, indent=2), encoding="utf-8")

print("✓ tutorias.json  — normativa CU-0220-2017-UNSAAC estructurada")
print("✓ practicas.json — PPP: requisitos, duración, normativa, titulación")
print("✓ servicios.json — BIE, MOV, SER: servicios institucionales")
print()
print("Repositorio listo en:", BASE.resolve())


✓ tutorias.json  — normativa CU-0220-2017-UNSAAC estructurada
✓ practicas.json — PPP: requisitos, duración, normativa, titulación
✓ servicios.json — BIE, MOV, SER: servicios institucionales

Repositorio listo en: /content/Repositorio_Conocimiento


In [17]:
# ═══════════════════════════════════════════════════
# CELDA 4 — Clase Preprocessor
# ═══════════════════════════════════════════════════
import re, unicodedata

class Preprocessor:
    """
    Normaliza el texto crudo del usuario:
    - lowercase
    - elimina tildes y diacríticos
    - elimina signos de puntuación
    - tokeniza
    - elimina stopwords en español
    """
    STOPWORDS = {
        "la","el","en","de","que","es","un","una","los","las","para","por","con",
        "se","del","al","si","me","mi","tu","su","hay","son","esta","puedo","debo",
        "tengo","como","cuando","donde","cual","cuales","cuantos","cuantas","mas",
        "pero","sin","sobre","entre","hasta","desde","hacia","siendo","siendo",
        "mis","sus","nos","les","les","ya","no","ni","yo","el","a","e","o","u",
        "le","lo","te","ti","ser","era","fue","han","han","has","he","hemos","estoy",
        "estan","esta","eres","poder","hacer","ver","dar","muy","bien","aqui","ahi"
    }

    def normalize(self, text: str) -> str:
        """Normaliza: minúsculas, sin tildes, sin puntuación."""
        text = text.lower().strip()
        # Eliminar tildes / diacríticos
        text = unicodedata.normalize("NFD", text)
        text = "".join(c for c in text if unicodedata.category(c) != "Mn")
        # Eliminar signos especiales (conservar letras, números, espacios)
        text = re.sub(r"[^\w\s]", " ", text)
        # Colapsar espacios múltiples
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def tokenize(self, text: str) -> list[str]:
        """Tokeniza y filtra stopwords y tokens cortos."""
        normalized = self.normalize(text)
        tokens = normalized.split()
        return [t for t in tokens if t not in self.STOPWORDS and len(t) > 2]

    def process(self, text: str) -> tuple[str, list[str]]:
        """Devuelve (texto_normalizado, tokens_limpios)."""
        normalized = self.normalize(text)
        tokens = self.tokenize(text)
        return normalized, tokens


In [18]:
# ═══════════════════════════════════════════════════
# CELDA 5 — Clase KnowledgeBaseLoader
# ═══════════════════════════════════════════════════
import json
from pathlib import Path

class KnowledgeBaseLoader:
    """
    Carga todos los JSON del repositorio en memoria al instanciarse.
    Expone métodos de acceso O(1) basados en índices preconstruidos.
    """

    def __init__(self, base_dir: str = "Repositorio_Conocimiento"):
        self._base = Path(base_dir)
        self._corpus: list[dict] = []
        self._intents: list[dict] = []
        self._keywords: list[dict] = []
        self._faq_index: dict = {}
        self._tutorias: dict = {}
        self._practicas: dict = {}
        self._servicios: dict = {}
        # Índices preconstruidos
        self._corpus_by_id: dict[str, dict] = {}
        self._corpus_by_intent: dict[str, dict] = {}
        self._keywords_by_intent: dict[str, dict] = {}
        self._load_all()

    def _load_json(self, path: str):
        full = self._base / path
        if not full.exists():
            raise FileNotFoundError(f"JSON no encontrado: {full}")
        with open(full, "r", encoding="utf-8") as f:
            return json.load(f)

    def _load_all(self):
        self._corpus      = self._load_json("corpus/corpus_general.json")
        intents_raw       = self._load_json("knowledge_base/intents.json")
        self._intents     = intents_raw.get("intents", intents_raw) if isinstance(intents_raw, dict) else intents_raw
        kw_raw            = self._load_json("knowledge_base/keywords.json")
        self._keywords    = kw_raw.get("keywords", kw_raw) if isinstance(kw_raw, dict) else kw_raw
        self._faq_index   = self._load_json("knowledge_base/faq_index.json")
        self._tutorias    = self._load_json("knowledge_files/tutorias.json")
        self._practicas   = self._load_json("knowledge_files/practicas.json")
        self._servicios   = self._load_json("knowledge_files/servicios.json")
        # Construir índices
        for e in self._corpus:
            self._corpus_by_id[e["id"]] = e
            self._corpus_by_intent[e["intent"]] = e
        for kw in self._keywords:
            self._keywords_by_intent[kw["intent_id"]] = kw
        print(f"  [KBLoader] {len(self._corpus)} entradas | "
              f"{len(self._intents)} intents | "
              f"{len(self._keywords)} keyword-entries cargadas")

    # ── Acceso O(1) ──────────────────────────────────────────────
    def get_by_intent(self, intent_id: str) -> dict | None:
        """Recupera entrada del corpus por intent_id. Complejidad O(1)."""
        # Primero intenta el mapa rápido del faq_index
        rapid = self._faq_index.get("busqueda_rapida", {})
        entry_id = rapid.get(intent_id)
        if entry_id:
            return self._corpus_by_id.get(entry_id)
        # Fallback al índice en memoria
        return self._corpus_by_intent.get(intent_id)

    def get_by_id(self, entry_id: str) -> dict | None:
        return self._corpus_by_id.get(entry_id)

    def get_by_category(self, cat_id: str) -> list[dict]:
        return [e for e in self._corpus if e["categoria"] == cat_id]

    def get_keywords(self, intent_id: str) -> dict | None:
        return self._keywords_by_intent.get(intent_id)

    def get_all_keywords(self) -> list[dict]:
        return self._keywords

    def get_normative_detail(self, categoria: str) -> dict | None:
        """Devuelve el archivo de conocimiento especializado según categoría."""
        if categoria in ("TUT", "TTUT"):
            return self._tutorias
        elif categoria == "PPP":
            return self._practicas
        elif categoria in ("BIE", "MOV", "SER"):
            return self._servicios
        return None

    def get_confianza_minima(self, intent_id: str) -> float:
        for i in self._intents:
            if i["intent_id"] == intent_id:
                return i.get("confianza_minima", 0.72)
        return 0.72

    @property
    def categories(self) -> list[str]:
        return sorted({e["categoria"] for e in self._corpus})


In [19]:
# ═══════════════════════════════════════════════════
# CELDA 6 — Clase IntentMatcher
# ═══════════════════════════════════════════════════

class IntentResult:
    """Resultado de la detección de intención."""
    def __init__(self, intent_id: str | None, score: float, categoria: str | None):
        self.intent_id = intent_id
        self.score     = score
        self.categoria = categoria
        self.matched   = intent_id is not None

    def __repr__(self):
        return f"IntentResult(intent='{self.intent_id}', score={self.score:.3f}, cat='{self.categoria}')"


class IntentMatcher:
    """
    Detecta la intención del usuario usando scoring ponderado de keywords.

    Algoritmo (dos fases):
    ──────────────────────
    Fase 1 — Trigger phrases: si el texto normalizado contiene una frase
             trigger exacta → score 1.0 para esa intención (alta confianza).

    Fase 2 — Scoring ponderado:
        score = (primarios_encontrados × 2 + secundarios_encontrados × 1
                 + sinonimos_encontrados × 1.5)
                ÷ (total_primarios × 2 + total_secundarios + total_sin × 1.5)

    Se elige el intent con mayor score que supere su confianza_minima.
    """
    THRESHOLD_FALLBACK = 0.12   # mínimo absoluto para no hacer fallback

    def __init__(self, kb: KnowledgeBaseLoader, preprocessor: Preprocessor):
        self._kb   = kb
        self._prep = preprocessor

    def _score_entry(self, kw_entry: dict, normalized: str, tokens: list[str]) -> float:
        """Calcula score ponderado para un intent dado el texto del usuario."""
        # ── Fase 1: trigger phrase ────────────────────────────────────────────
        for trigger in kw_entry.get("frases_trigger", []):
            t_norm = self._prep.normalize(trigger)
            if t_norm in normalized:
                return 1.0   # coincidencia exacta de frase trigger

        # ── Fase 2: scoring ponderado ─────────────────────────────────────────
        def token_match(kw: str) -> bool:
            kw_n = self._prep.normalize(kw)
            # coincidencia si la keyword normalizada aparece en el texto
            # o si algún token del usuario está contenido en la keyword
            if kw_n in normalized:
                return True
            kw_tokens = kw_n.split()
            return any(t in kw_tokens or kt in kw_n for t in tokens for kt in kw_tokens)

        primarios    = kw_entry.get("keywords_primarios", [])
        secundarios  = kw_entry.get("keywords_secundarios", [])
        sinonimos    = kw_entry.get("sinonimos", [])

        pm = sum(1 for k in primarios   if token_match(k))
        sm = sum(1 for k in secundarios if token_match(k))
        yn = sum(1 for k in sinonimos   if token_match(k))

        max_score = len(primarios)*2.0 + len(secundarios)*1.0 + len(sinonimos)*1.5
        if max_score == 0:
            return 0.0
        raw = pm*2.0 + sm*1.0 + yn*1.5
        return raw / max_score

    def detect(self, user_text: str) -> IntentResult:
        """
        Detecta la intención del usuario.
        Devuelve IntentResult con intent_id=None si no hay coincidencia suficiente.
        """
        normalized, tokens = self._prep.process(user_text)
        if not tokens:
            return IntentResult(None, 0.0, None)

        best_intent  = None
        best_score   = 0.0
        best_cat     = None

        for kw_entry in self._kb.get_all_keywords():
            score = self._score_entry(kw_entry, normalized, tokens)
            if score > best_score:
                best_score  = score
                best_intent = kw_entry["intent_id"]
                best_cat    = kw_entry.get("categoria")

        # Verificar umbral
        if best_intent and best_score >= self.THRESHOLD_FALLBACK:
            # Verificar también la confianza mínima específica del intent
            conf_min = self._kb.get_confianza_minima(best_intent)
            if best_score >= conf_min * 0.5:   # umbral relajado para prototipo
                return IntentResult(best_intent, best_score, best_cat)

        return IntentResult(None, best_score, None)


In [20]:
# ═══════════════════════════════════════════════════
# CELDA 7 — Clase ResponseBuilder
# ═══════════════════════════════════════════════════

class ResponseBuilder:
    """
    Construye la respuesta final a partir del IntentResult.

    Niveles de construcción:
    ─────────────────────────
    1. Respuesta directa del corpus (campo 'respuesta')
    2. Enriquecimiento normativo según categoría:
         TUT/TTUT → referencia a artículo específico de tutorias.json
         PPP      → referencia normativa de practicas.json
         BIE/MOV/SER → datos de acceso de servicios.json
    3. Fallback con menú de categorías disponibles
    """

    CAT_LABELS = {
        "TUT":"Proceso de Tutorías","TTUT":"Tipos de Tutoría",
        "CUR":"Cursos y Semestres","ESP":"Especialidades",
        "PPP":"Prácticas Pre-Profesionales","BIE":"Bienestar Universitario",
        "MOV":"Movilidad Estudiantil","MAT":"Matrícula",
        "TIT":"Titulación","SER":"Servicios Universitarios"
    }

    def __init__(self, kb: KnowledgeBaseLoader):
        self._kb = kb

    def build(self, intent_result: IntentResult) -> str:
        if not intent_result.matched:
            return self._fallback()
        entry = self._kb.get_by_intent(intent_result.intent_id)
        if not entry:
            return self._fallback()
        # ── Nivel 1: respuesta base ───────────────────────────────────────────
        respuesta = entry["respuesta"]
        # ── Nivel 2: enriquecimiento normativo ────────────────────────────────
        enriq = self._enriquecimiento(entry)
        if enriq:
            respuesta += "\n" + enriq
        # ── Pie de fuente ─────────────────────────────────────────────────────
        respuesta += f"\n\n📌 Fuente: {entry.get('fuente','—')}"
        return respuesta

    def _enriquecimiento(self, entry: dict) -> str:
        """Agrega información normativa adicional si la categoría lo permite."""
        cat  = entry.get("categoria","")
        norm = self._kb.get_normative_detail(cat)
        if not norm:
            return ""
        if cat in ("TUT","TTUT"):
            # Añadir referencia a confidencialidad o asignación si aplica
            if "confidencial" in entry["intent"]:
                conf = norm.get("confidencialidad",{})
                return f"ℹ️  {conf.get('art','')} — {conf.get('deber','')}"
            if "frecuencia" in entry["intent"]:
                per = norm.get("periodicidad",{})
                momentos = per.get("momentos_clave_semestre",[])
                if momentos:
                    return "📅 Momentos clave: " + " | ".join(momentos)
            if "cambio_tutor" in entry["intent"]:
                asig = norm.get("asignacion_tutores",{})
                return f"ℹ️  {asig.get('art','')} — Máx. tutorados por docente: {asig.get('maximo_tutorados',25)}"
        elif cat == "PPP":
            if "remuneracion" in entry["intent"]:
                rem = norm.get("remuneracion",{})
                return f"⚖️  {rem.get('norma','')} — Subvención mínima: {rem.get('subvencion_minima','')}"
            if "requisito" in entry["intent"] and "titulacion" in entry["intent"]:
                rel = norm.get("relacion_titulacion",{})
                return f"🎓 {rel.get('descripcion','')}"
        elif cat == "MOV":
            mods = norm.get("modulos",{}).get("movilidad_estudiantil",{})
            programas = mods.get("programas",[])
            if programas and "programas" in entry["intent"]:
                return "🌍 Programas: " + " | ".join(programas[:3]) + "..."
        return ""

    def _fallback(self) -> str:
        cats = "\n  ".join(f"• {v}" for k,v in self.CAT_LABELS.items())
        return (
            "Lo siento, no pude identificar tu consulta con certeza.\n"
            "Puedes preguntarme sobre:\n  " + cats +
            "\n\n💡 Tip: sé más específico. Ej.: '¿Qué es la tutoría académica?' "
            "o '¿Cuáles son los requisitos para las prácticas?'"
        )

    def header(self, entry: dict) -> str:
        """Devuelve el encabezado con pregunta canónica (para depuración)."""
        return f"[{entry.get('id','?')} | {entry.get('categoria_nombre','?')}]\n❓ {entry.get('pregunta','?')}"


In [21]:
# ═══════════════════════════════════════════════════
# CELDA 8 — Clase ConversationHistory
# ═══════════════════════════════════════════════════
from datetime import datetime

class ConversationHistory:
    """
    Registra turnos de conversación y genera reporte de evaluación.

    Estructura de un turno:
    {timestamp, user_input, normalized, intent_id, score,
     categoria, respuesta_entregada, fue_fallback}
    """

    def __init__(self):
        self._turnos: list[dict] = []
        self._session_start = datetime.now()

    def record(self, user_input: str, normalized: str,
               intent_result: IntentResult, respuesta: str):
        self._turnos.append({
            "timestamp":  datetime.now().isoformat(timespec="seconds"),
            "user_input": user_input,
            "normalized": normalized,
            "intent_id":  intent_result.intent_id,
            "score":      round(intent_result.score, 4),
            "categoria":  intent_result.categoria,
            "fue_fallback": not intent_result.matched
        })

    @property
    def total_turnos(self) -> int:
        return len(self._turnos)

    @property
    def tasa_resolucion(self) -> float:
        if not self._turnos:
            return 0.0
        resueltos = sum(1 for t in self._turnos if not t["fue_fallback"])
        return round(resueltos / len(self._turnos) * 100, 1)

    @property
    def tasa_fallback(self) -> float:
        return round(100 - self.tasa_resolucion, 1)

    def distribucion_categorias(self) -> dict[str, int]:
        dist = {}
        for t in self._turnos:
            if t["categoria"]:
                dist[t["categoria"]] = dist.get(t["categoria"], 0) + 1
        return dict(sorted(dist.items(), key=lambda x: -x[1]))

    def show(self):
        """Imprime historial de la sesión."""
        print("=" * 60)
        print(f"  HISTORIAL DE SESIÓN — {self._session_start.strftime('%Y-%m-%d %H:%M')}")
        print("=" * 60)
        for i, t in enumerate(self._turnos, 1):
            estado = "✓" if not t["fue_fallback"] else "✗"
            print(f"  [{i:02d}] {estado} [{t['intent_id'] or 'FALLBACK':45s}] score={t['score']:.3f}")
            print(f"       Usuario: {t['user_input'][:70]}")
        print("-" * 60)

    def export_report(self) -> str:
        """Genera reporte de evaluación para el artículo científico."""
        dist = self.distribucion_categorias()
        lines = [
            "╔══════════════════════════════════════════════════════════╗",
            "║         REPORTE DE EVALUACIÓN — CHATBOT EPIIS-UNSAAC    ║",
            "╚══════════════════════════════════════════════════════════╝",
            f"  Sesión iniciada : {self._session_start.strftime('%Y-%m-%d %H:%M:%S')}",
            f"  Total turnos    : {self.total_turnos}",
            f"  Tasa resolución : {self.tasa_resolucion}%",
            f"  Tasa fallback   : {self.tasa_fallback}%",
            "",
            "  Distribución por categoría:",
        ]
        for cat, cnt in dist.items():
            pct = round(cnt / max(self.total_turnos, 1) * 100, 1)
            bar = "█" * int(pct / 5)
            lines.append(f"    {cat:8s} {bar:<20s} {cnt:3d} ({pct}%)")
        lines += ["", "  Detalle por turno:"]
        for i, t in enumerate(self._turnos, 1):
            estado = "RESUELTO" if not t["fue_fallback"] else "FALLBACK"
            lines.append(f"  [{i:02d}] {estado:8s} | {t['intent_id'] or '—':40s} | score={t['score']:.3f}")
        return "\n".join(lines)


In [22]:
# ═══════════════════════════════════════════════════
# CELDA 9 — Clase ChatbotEPIIS (Fachada principal)
# ═══════════════════════════════════════════════════

class ChatbotEPIIS:
    """
    Fachada del sistema. Única clase que el notebook usa directamente.

    Instancia y coordina internamente:
      Preprocessor → IntentMatcher → KnowledgeBaseLoader → ResponseBuilder
      → ConversationHistory

    Uso básico:
        bot = ChatbotEPIIS()
        print(bot.respond("¿Qué es la tutoría?"))
        bot.show_history()
        print(bot.export_report())
    """

    BANNER = """
╔══════════════════════════════════════════════════════════════╗
║       🤖  CHATBOT ACADÉMICO EPIIS-UNSAAC  —  v1.0           ║
║  Puedo responder consultas sobre tutorías, matrícula,        ║
║  titulación, prácticas, especialidades y servicios UNSAAC.   ║
║  Escribe 'ayuda' para ver categorías · 'salir' para terminar ║
╚══════════════════════════════════════════════════════════════╝"""

    HELP_TEXT = """
Puedes consultarme sobre:
  📚 TUT  — Proceso de Tutorías
  📌 TTUT — Tipos de Tutoría
  📖 CUR  — Cursos y Semestres
  🎯 ESP  — Especialidades de la Escuela
  💼 PPP  — Prácticas Pre-Profesionales
  🏥 BIE  — Bienestar Universitario
  ✈️  MOV  — Movilidad Estudiantil
  📋 MAT  — Matrícula
  🎓 TIT  — Titulación
  🖥️  SER  — Servicios Universitarios

Ejemplos:
  → ¿Qué es la tutoría académica?
  → ¿Cuántos créditos necesito para graduarme?
  → ¿Cuáles son los requisitos para las prácticas?
  → ¿Cómo me matriculo en SERUNSA?
  → ¿Cuáles son las modalidades de titulación?"""

    def __init__(self, base_dir: str = "Repositorio_Conocimiento"):
        print("Inicializando Chatbot EPIIS-UNSAAC...")
        self._preprocessor = Preprocessor()
        self._kb            = KnowledgeBaseLoader(base_dir)
        self._matcher       = IntentMatcher(self._kb, self._preprocessor)
        self._builder       = ResponseBuilder(self._kb)
        self._history       = ConversationHistory()
        print("✓ Sistema listo.\n")

    def respond(self, user_input: str, *, verbose: bool = False) -> str:
        """
        Procesa la entrada del usuario y devuelve la respuesta.
        Si verbose=True imprime también el IntentResult para depuración.
        """
        if not user_input.strip():
            return "Por favor, escribe tu consulta."

        cmd = user_input.strip().lower()
        if cmd in ("ayuda","help","?","que puedes hacer"):
            return self.HELP_TEXT
        if cmd in ("historial","historia","turnos"):
            self._history.show()
            return ""
        if cmd in ("reporte","report","evaluacion"):
            return self._history.export_report()

        normalized, _ = self._preprocessor.process(user_input)
        result = self._matcher.detect(user_input)

        if verbose:
            print(f"  🔍 {result}")

        respuesta = self._builder.build(result)
        self._history.record(user_input, normalized, result, respuesta)
        return respuesta

    def start_session(self):
        print(self.BANNER)

    def show_history(self):
        self._history.show()

    def export_report(self) -> str:
        return self._history.export_report()

    @property
    def stats(self) -> dict:
        return {
            "turnos": self._history.total_turnos,
            "tasa_resolucion": self._history.tasa_resolucion,
            "tasa_fallback": self._history.tasa_fallback,
            "distribucion": self._history.distribucion_categorias()
        }


In [23]:
# ═══════════════════════════════════════════════════
# CELDA 10 — Inicialización y verificación del sistema
# ═══════════════════════════════════════════════════
bot = ChatbotEPIIS()
bot.start_session()

# Verificación rápida del repositorio
print(f"Categorías disponibles : {bot._kb.categories}")
print(f"Entradas corpus        : {len(bot._kb._corpus)}")
print(f"Keywords cargadas      : {len(bot._kb._keywords)}")
print(f"Mapa faq_index         : {len(bot._kb._faq_index.get('busqueda_rapida',{}))} intents")


Inicializando Chatbot EPIIS-UNSAAC...
  [KBLoader] 100 entradas | 100 intents | 100 keyword-entries cargadas
✓ Sistema listo.


╔══════════════════════════════════════════════════════════════╗
║       🤖  CHATBOT ACADÉMICO EPIIS-UNSAAC  —  v1.0           ║
║  Puedo responder consultas sobre tutorías, matrícula,        ║
║  titulación, prácticas, especialidades y servicios UNSAAC.   ║
║  Escribe 'ayuda' para ver categorías · 'salir' para terminar ║
╚══════════════════════════════════════════════════════════════╝
Categorías disponibles : ['BIE', 'CUR', 'ESP', 'MAT', 'MOV', 'PPP', 'SER', 'TIT', 'TTUT', 'TUT']
Entradas corpus        : 100
Keywords cargadas      : 100
Mapa faq_index         : 100 intents


In [24]:
# ═══════════════════════════════════════════════════
# CELDA 11 — Casos de prueba automáticos
# ═══════════════════════════════════════════════════

CASOS_PRUEBA = [
    # (entrada_usuario, intent_esperado)
    # ── TUT ──
    ("¿Qué es la tutoría académica en la UNSAAC?",          "consulta_tutoria_definicion"),
    ("¿Con qué frecuencia debo asistir a tutoría?",          "consulta_tutoria_frecuencia"),
    ("¿Las sesiones de tutoría son confidenciales?",         "consulta_tutoria_confidencialidad"),
    ("¿Puedo cambiar de tutor?",                             "consulta_tutoria_cambio_tutor"),
    # ── TTUT ──
    ("cuales son los tipos de tutoria",                      "consulta_tipos_tutoria_general"),
    ("hay tutoria virtual o en linea",                       "consulta_tipos_tutoria_virtual"),
    ("tutoría para estudiantes con bajo rendimiento",        "consulta_tipos_tutoria_riesgo_academico"),
    ("qué diferencia hay entre tutor y asesor de tesis",     "consulta_tipos_tutoria_vs_asesoria_tesis"),
    # ── CUR ──
    ("cuantos semestres dura la carrera de sistemas",        "consulta_cursos_duracion_carrera"),
    ("cuantos creditos necesito para graduarme",             "consulta_cursos_total_creditos"),
    ("qué pasa si jalo un curso",                            "consulta_cursos_desaprobacion"),
    ("cómo se calcula el promedio ponderado",                "consulta_cursos_promedio_ponderado"),
    # ── ESP ──
    ("que especialidades tiene la epiis",                    "consulta_esp_listado_especialidades"),
    ("cuando tengo que elegir mi especialidad",              "consulta_esp_cuando_elegir"),
    ("que se estudia en inteligencia artificial y datos",    "consulta_esp_ia_ciencia_datos"),
    # ── PPP ──
    ("cuales son los requisitos para las practicas ppp",     "consulta_ppp_requisitos_inicio"),
    ("cuantas horas de practicas debo hacer",                "consulta_ppp_horas_acumuladas"),
    ("me pagan en las practicas pre profesionales",          "consulta_ppp_remuneracion"),
    ("practicas son requisito para titularme",               "consulta_ppp_requisito_titulacion"),
    # ── BIE ──
    ("que servicios de bienestar ofrece la unsaac",          "consulta_bie_servicios_generales"),
    ("como accedo al psicologo gratuito",                    "consulta_bie_atencion_psicologica"),
    ("como solicito la beca de alimentacion",                "consulta_bie_beca_alimentacion"),
    # ── MOV ──
    ("que programas de intercambio hay",                     "consulta_mov_programas_disponibles"),
    ("requisitos para el intercambio estudiantil",           "consulta_mov_requisitos_intercambio"),
    ("me reconocen los cursos del intercambio",              "consulta_mov_convalidacion_cursos"),
    # ── MAT ──
    ("cuando es la matricula en la unsaac",                  "consulta_mat_fechas_matricula"),
    ("como me matriculo en serunsa",                         "consulta_mat_proceso_serunsa"),
    ("que es la matricula condicional",                      "consulta_mat_condicional"),
    ("cuantos creditos maximo puedo llevar",                 "consulta_mat_creditos_maximo"),
    # ── TIT ──
    ("cuales son las modalidades de titulacion",             "consulta_tit_modalidades"),
    ("como obtengo el grado de bachiller",                   "consulta_tit_grado_bachiller"),
    ("cuanto tarda el proceso de titulacion",                "consulta_tit_duracion_proceso"),
    ("donde registro mi titulo en sunedu",                   "consulta_tit_registro_sunedu"),
    # ── SER ──
    ("como accedo al aula virtual moodle",                   "consulta_ser_aula_virtual"),
    ("como saco mi correo institucional",                    "consulta_ser_correo_institucional"),
    ("que es el tupa",                                       "consulta_ser_tupa"),
    # ── VARIACIONES LÉXICAS ──
    ("prf cuándo es la matrí",                               None),   # muy corta → fallback esperado
    ("necesito saber sobre las ppp",                         "consulta_ppp_requisitos_inicio"),
    ("jale un curso qué hago",                               "consulta_cursos_desaprobacion"),
    ("quiero cambiar de tutor xq no me llevo",               "consulta_tutoria_cambio_tutor"),
]

# ── Ejecución ────────────────────────────────────────────────────
aciertos = 0
errores  = 0
fallbacks_esperados = 0
fallbacks_ok = 0

print(f"{'#':>3}  {'ESTADO':8}  {'SCORE':6}  {'ESPERADO':40}  {'OBTENIDO':40}")
print("-" * 110)

for i, (consulta, esperado) in enumerate(CASOS_PRUEBA, 1):
    resultado = bot._matcher.detect(consulta)
    obtenido  = resultado.intent_id

    if esperado is None:
        # Caso fallback esperado
        fallbacks_esperados += 1
        ok = obtenido is None
        if ok: fallbacks_ok += 1
        estado = "✓ FB-OK" if ok else "✗ FB-ERR"
    else:
        ok     = (obtenido == esperado)
        estado = "✓ OK   " if ok else "✗ FAIL "
        if ok:   aciertos += 1
        else:    errores  += 1

    print(f"{i:>3}  {estado}  {resultado.score:6.3f}  {str(esperado):40s}  {str(obtenido):40s}")

total_con_esperado = len([c for c in CASOS_PRUEBA if c[1] is not None])
print("=" * 110)
print(f"\n  RESUMEN DE PRUEBAS AUTOMÁTICAS")
print(f"  Total casos con intent esperado  : {total_con_esperado}")
print(f"  Aciertos                         : {aciertos} ({aciertos/max(total_con_esperado,1)*100:.1f}%)")
print(f"  Errores                          : {errores}")
print(f"  Fallbacks esperados/correctos    : {fallbacks_ok}/{fallbacks_esperados}")
print(f"  Tasa de resolución global        : {bot.stats['tasa_resolucion']}%")


  #  ESTADO    SCORE   ESPERADO                                  OBTENIDO                                
--------------------------------------------------------------------------------------------------------------
  1  ✓ OK      1.000  consulta_tutoria_definicion               consulta_tutoria_definicion             
  2  ✗ FAIL    1.000  consulta_tutoria_frecuencia               consulta_tutoria_definicion             
  3  ✗ FAIL    1.000  consulta_tutoria_confidencialidad         consulta_tutoria_definicion             
  4  ✗ FAIL    1.000  consulta_tutoria_cambio_tutor             consulta_tutoria_definicion             
  5  ✗ FAIL    1.000  consulta_tipos_tutoria_general            consulta_tutoria_definicion             
  6  ✗ FAIL    1.000  consulta_tipos_tutoria_virtual            consulta_tutoria_definicion             
  7  ✗ FAIL    1.000  consulta_tipos_tutoria_riesgo_academico   consulta_tutoria_definicion             
  8  ✗ FAIL    1.000  consulta_tipos_tutoria_vs_

## 💬 Chat Interactivo

Ejecuta la celda siguiente para iniciar el chat.  
Escribe cualquier consulta académica. Comandos especiales:
- `ayuda` → ver categorías disponibles
- `historial` → ver turnos de la sesión  
- `reporte` → exportar reporte de evaluación
- `salir` → terminar el chat


In [25]:
# ═══════════════════════════════════════════════════
# CELDA 12 — Chat interactivo
# ═══════════════════════════════════════════════════
print("\n" + "="*62)
print("  MODO CHAT ACTIVO — escribe 'salir' para terminar")
print("="*62 + "\n")

while True:
    try:
        entrada = input("Tú: ").strip()
    except EOFError:
        # En entornos no interactivos se sale automáticamente
        break

    if not entrada:
        continue
    if entrada.lower() in ("salir","exit","quit","bye"):
        print("\nBot: ¡Hasta luego! Que tengas un excelente semestre. 🎓")
        print("\n--- Reporte final de sesión ---")
        print(bot.export_report())
        break

    respuesta = bot.respond(entrada)
    if respuesta:
        print(f"\nBot: {respuesta}\n")



  MODO CHAT ACTIVO — escribe 'salir' para terminar



KeyboardInterrupt: Interrupted by user